In [2]:
# ============================================
# Cell 1: Write Phase 5 adaptive model runner
# ============================================

from pathlib import Path

ROOT = Path("/data/Sajjan_Singh/spml/wavelet_seq_project")
SCRIPTS_DIR = ROOT / "scripts"
SCRIPTS_DIR.mkdir(parents=True, exist_ok=True)

runner_path = SCRIPTS_DIR / "phase5_adaptive_wavelet_model.py"

runner_code = r'''
import gc
import json
import math
import random
import time
from pathlib import Path
from typing import Dict, List, Optional

import numpy as np
import pandas as pd
import pywt
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

ROOT = Path("/data/Sajjan_Singh/spml/wavelet_seq_project")
PROCESSED_ROOT = ROOT / "data" / "processed"
SPLIT_ROOT = ROOT / "data" / "splits"
PHASE5_TABLES = ROOT / "results" / "tables" / "phase5"
PHASE5_TABLES.mkdir(parents=True, exist_ok=True)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if hasattr(torch, "set_float32_matmul_precision"):
    torch.set_float32_matmul_precision("high")

ALL_DATASETS = ["etth1", "etth2", "ettm1", "ettm2", "weather", "electricity", "traffic", "exchange", "ili"]

LONG_PRED = {
    "etth1": 96,
    "etth2": 96,
    "ettm1": 96,
    "ettm2": 96,
    "weather": 96,
    "electricity": 96,
    "traffic": 96,
    "exchange": 96,
    "ili": 24,
}

SEARCH_LOOKBACK_CANDIDATES = {
    "etth1": [48, 96, 168],
    "etth2": [48, 96, 168],
    "ettm1": [96, 192, 336],
    "ettm2": [96, 192, 336],
    "weather": [96, 192, 288],
    "electricity": [96, 168, 336],
    "traffic": [96, 168, 336],
    "exchange": [48, 96, 192],
    "ili": [26, 52, 104],
}

DEFAULT_BATCH_SIZE = {
    "etth1": 256,
    "etth2": 256,
    "ettm1": 256,
    "ettm2": 256,
    "weather": 192,
    "electricity": 128,
    "traffic": 96,
    "exchange": 256,
    "ili": 64,
}

DEFAULT_EPOCHS = {
    "etth1": 30,
    "etth2": 30,
    "ettm1": 30,
    "ettm2": 30,
    "weather": 30,
    "electricity": 30,
    "traffic": 30,
    "exchange": 30,
    "ili": 35,
}

FAMILY_MAP = {
    "Haar": "haar",
    "Db2": "db2",
    "Db4": "db4",
    "Symlet": "sym4",
    "Coiflet": "coif1",
    "haar": "haar",
    "db2": "db2",
    "db4": "db4",
    "sym4": "sym4",
    "coif1": "coif1",
}

TRAIN_CFG = {
    "lr": 1e-3,
    "weight_decay": 1e-5,
    "dropout": 0.1,
    "proj_dim": 128,
    "hidden_dim": 128,
    "num_layers": 2,
    "search_epochs": 8,
    "search_patience": 3,
    "full_patience": 5,
    "num_workers": 2,
    "use_amp": torch.cuda.is_available(),
    "grad_clip": 1.0,
}

# ------------------------------------------------------------
# Utilities
# ------------------------------------------------------------

def set_seed(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

def compute_metrics(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=np.float64)
    y_pred = np.asarray(y_pred, dtype=np.float64)
    mse = float(np.mean((y_true - y_pred) ** 2))
    mae = float(np.mean(np.abs(y_true - y_pred)))
    rmse = float(np.sqrt(mse))
    return {"mse": mse, "mae": mae, "rmse": rmse}

def load_bundle(dataset_name: str, input_mode: str = "multivariate"):
    npz_path = PROCESSED_ROOT / dataset_name / f"{dataset_name}_prepared.npz"
    split_path = SPLIT_ROOT / f"{dataset_name}_splits.json"

    data = np.load(npz_path, allow_pickle=True)
    with open(split_path, "r") as f:
        splits = json.load(f)

    values_raw = data["values_raw"].astype(np.float32)
    values_scaled = data["values_scaled"].astype(np.float32)
    scaler_mean = data["scaler_mean"].astype(np.float32)
    scaler_std = data["scaler_std"].astype(np.float32)
    target_idx = int(data["target_idx"][0])

    if input_mode == "univariate":
        values_raw = values_raw[:, target_idx:target_idx+1]
        values_scaled = values_scaled[:, target_idx:target_idx+1]
        scaler_mean = scaler_mean[target_idx:target_idx+1]
        scaler_std = scaler_std[target_idx:target_idx+1]
        target_idx = 0

    return {
        "values_raw": values_raw,
        "values_scaled": values_scaled,
        "scaler_mean": scaler_mean,
        "scaler_std": scaler_std,
        "target_idx": target_idx,
        "splits": {k: int(v) for k, v in splits.items()},
    }

class ForecastWindowDataset(Dataset):
    def __init__(self, bundle, split_name: str, seq_len: int, pred_len: int):
        self.values_raw = bundle["values_raw"]
        self.values_scaled = bundle["values_scaled"]
        self.target_idx = bundle["target_idx"]
        self.seq_len = seq_len
        self.pred_len = pred_len

        s = bundle["splits"]
        if split_name == "train":
            self.border1 = s["train_start"]
            self.border2 = s["train_end"]
        elif split_name == "val":
            self.border1 = max(0, s["val_start"] - seq_len)
            self.border2 = s["val_end"]
        elif split_name == "test":
            self.border1 = max(0, s["test_start"] - seq_len)
            self.border2 = s["test_end"]
        else:
            raise ValueError(split_name)

        self.length = self.border2 - self.border1 - seq_len - pred_len + 1
        if self.length <= 0:
            raise ValueError(f"Invalid windowing for split={split_name}, seq_len={seq_len}, pred_len={pred_len}")

    def __len__(self):
        return self.length

    def __getitem__(self, idx):
        s = self.border1 + idx
        x_scaled = self.values_scaled[s:s+self.seq_len]
        y_scaled = self.values_scaled[s+self.seq_len:s+self.seq_len+self.pred_len, self.target_idx]
        y_raw = self.values_raw[s+self.seq_len:s+self.seq_len+self.pred_len, self.target_idx]
        return {
            "x_scaled": torch.tensor(x_scaled, dtype=torch.float32),
            "y_scaled": torch.tensor(y_scaled, dtype=torch.float32),
            "y_raw": torch.tensor(y_raw, dtype=torch.float32),
        }

def make_loaders(bundle, seq_len: int, pred_len: int, batch_size: int):
    train_ds = ForecastWindowDataset(bundle, "train", seq_len, pred_len)
    val_ds = ForecastWindowDataset(bundle, "val", seq_len, pred_len)
    test_ds = ForecastWindowDataset(bundle, "test", seq_len, pred_len)

    train_loader = DataLoader(
        train_ds, batch_size=batch_size, shuffle=True,
        num_workers=TRAIN_CFG["num_workers"], pin_memory=torch.cuda.is_available(), drop_last=False
    )
    val_loader = DataLoader(
        val_ds, batch_size=batch_size, shuffle=False,
        num_workers=TRAIN_CFG["num_workers"], pin_memory=torch.cuda.is_available(), drop_last=False
    )
    test_loader = DataLoader(
        test_ds, batch_size=batch_size, shuffle=False,
        num_workers=TRAIN_CFG["num_workers"], pin_memory=torch.cuda.is_available(), drop_last=False
    )
    return train_loader, val_loader, test_loader

def resolve_pred_len(dataset_name: str, horizon_mode: str):
    if horizon_mode == "long":
        return LONG_PRED[dataset_name]
    raise ValueError(f"Unsupported horizon_mode: {horizon_mode}")

def resolve_seq_candidates(dataset_name: str, lookback_mode: str):
    if lookback_mode == "searched":
        return SEARCH_LOOKBACK_CANDIDATES[dataset_name]
    elif lookback_mode == "fixed":
        return [SEARCH_LOOKBACK_CANDIDATES[dataset_name][1]]
    raise ValueError(f"Unsupported lookback_mode: {lookback_mode}")

# ------------------------------------------------------------
# Phase 5 model components
# ------------------------------------------------------------

def build_wavelet_filters(wavelet_name: str):
    wavelet_name = FAMILY_MAP.get(wavelet_name, wavelet_name)
    wave = pywt.Wavelet(wavelet_name)
    dec_lo = np.array(wave.dec_lo[::-1], dtype=np.float32)
    dec_hi = np.array(wave.dec_hi[::-1], dtype=np.float32)
    return dec_lo, dec_hi

class AdaptiveWaveletDecomposer(nn.Module):
    """
    Analytic wavelet initialization + learnable filter bank + adaptive level gating.
    """
    def __init__(
        self,
        d_model: int,
        wavelet_family: str = "db4",
        levels: int = 3,
        filter_reg_lambda: float = 1e-4,
        gate_temperature: float = 1.0,
        gate_entropy_lambda: float = 1e-4,
    ):
        super().__init__()
        self.d_model = d_model
        self.wavelet_family = FAMILY_MAP.get(wavelet_family, wavelet_family)
        self.levels = levels
        self.filter_reg_lambda = filter_reg_lambda
        self.gate_temperature = gate_temperature
        self.gate_entropy_lambda = gate_entropy_lambda

        dec_lo, dec_hi = build_wavelet_filters(self.wavelet_family)
        k = len(dec_lo)

        init_low = torch.tensor(dec_lo, dtype=torch.float32).view(1, 1, -1).repeat(d_model, 1, 1)
        init_high = torch.tensor(dec_hi, dtype=torch.float32).view(1, 1, -1).repeat(d_model, 1, 1)

        self.register_buffer("init_low", init_low)
        self.register_buffer("init_high", init_high)

        self.low_filters = nn.ParameterList()
        self.high_filters = nn.ParameterList()
        for _ in range(levels):
            self.low_filters.append(nn.Parameter(init_low.clone()))
            self.high_filters.append(nn.Parameter(init_high.clone()))

        self.gate_net = nn.Sequential(
            nn.LayerNorm(d_model),
            nn.Linear(d_model, d_model),
            nn.GELU(),
            nn.Linear(d_model, levels + 1),
        )

    def _pad(self, x, kernel_size: int):
        pad_total = kernel_size - 1
        left = pad_total // 2
        right = pad_total - left
        return F.pad(x, (left, right), mode="replicate")

    def _conv_down(self, x, filt):
        x = self._pad(x, filt.shape[-1])
        return F.conv1d(x, filt, stride=2, groups=self.d_model)

    def _upsample_to(self, x, target_len: int):
        return F.interpolate(x, size=target_len, mode="nearest")

    def forward(self, x):
        # x: [B, T, D]
        B, T, D = x.shape
        current = x.transpose(1, 2)   # [B, D, T]

        # Adaptive level gating from global context
        context = x.mean(dim=1)       # [B, D]
        gate_logits = self.gate_net(context) / self.gate_temperature
        gates = torch.softmax(gate_logits, dim=-1)   # [B, levels+1]

        high_components = []
        for lvl in range(self.levels):
            low = self._conv_down(current, self.low_filters[lvl])
            high = self._conv_down(current, self.high_filters[lvl])

            high_up = self._upsample_to(high, T).transpose(1, 2)   # [B, T, D]
            gate = gates[:, lvl].view(B, 1, 1)
            high_components.append(gate * high_up)

            current = low

        final_low_up = self._upsample_to(current, T).transpose(1, 2)
        final_gate = gates[:, -1].view(B, 1, 1)
        final_low_up = final_gate * final_low_up

        return {
            "high_components": high_components,
            "final_low": final_low_up,
            "gates": gates,
        }

    def regularization_loss(self):
        reg = torch.tensor(0.0, device=self.low_filters[0].device)

        # toward analytic filters
        for low, high in zip(self.low_filters, self.high_filters):
            reg = reg + ((low - self.init_low.to(low.device)) ** 2).mean()
            reg = reg + ((high - self.init_high.to(high.device)) ** 2).mean()

            # encourage low/high separation (soft orthogonality)
            reg = reg + 0.1 * torch.mean((low * high) ** 2)

        reg = self.filter_reg_lambda * reg
        return reg

    def gate_entropy_loss(self, gates):
        # lower entropy -> more selective adaptive level choice
        eps = 1e-8
        entropy = -(gates * torch.log(gates + eps)).sum(dim=-1).mean()
        return self.gate_entropy_lambda * entropy

class CrossScaleFusion(nn.Module):
    """
    Cross-scale fusion using learnable scale attention + residual fusion.
    """
    def __init__(self, d_model: int, n_scales: int):
        super().__init__()
        self.scale_score = nn.Sequential(
            nn.LayerNorm(d_model),
            nn.Linear(d_model, d_model),
            nn.GELU(),
            nn.Linear(d_model, 1),
        )
        self.fuse = nn.Sequential(
            nn.LayerNorm(d_model * 2),
            nn.Linear(d_model * 2, d_model),
            nn.GELU(),
            nn.Linear(d_model, d_model),
        )
        self.out_norm = nn.LayerNorm(d_model)
        self.n_scales = n_scales

    def forward(self, raw_proj, high_components, final_low):
        # stack scales: raw + highs + final_low
        scales = [raw_proj] + list(high_components) + [final_low]  # each [B, T, D]
        stacked = torch.stack(scales, dim=2)  # [B, T, S, D]

        # score each scale using global time summary
        scale_context = stacked.mean(dim=1)   # [B, S, D]
        scores = self.scale_score(scale_context).squeeze(-1)  # [B, S]
        attn = torch.softmax(scores, dim=-1)                  # [B, S]

        fused_weighted = (stacked * attn.unsqueeze(1).unsqueeze(-1)).sum(dim=2)  # [B, T, D]
        fused = torch.cat([raw_proj, fused_weighted], dim=-1)
        fused = self.fuse(fused)
        return self.out_norm(raw_proj + fused), attn

class TemporalChannelMixerBlock(nn.Module):
    """
    Lightweight modern backbone block:
    - temporal depthwise conv mixer
    - channel MLP
    """
    def __init__(self, d_model: int, dropout: float = 0.1, kernel_size: int = 5):
        super().__init__()
        self.norm1 = nn.LayerNorm(d_model)
        self.dwconv = nn.Conv1d(
            in_channels=d_model,
            out_channels=d_model,
            kernel_size=kernel_size,
            padding=kernel_size // 2,
            groups=d_model,
        )
        self.pwconv = nn.Conv1d(
            in_channels=d_model,
            out_channels=d_model,
            kernel_size=1,
        )
        self.drop1 = nn.Dropout(dropout)

        self.norm2 = nn.LayerNorm(d_model)
        self.ffn = nn.Sequential(
            nn.Linear(d_model, d_model * 4),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d_model * 4, d_model),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        # x: [B, T, D]
        y = self.norm1(x)
        y = y.transpose(1, 2)
        y = self.dwconv(y)
        y = self.pwconv(y)
        y = y.transpose(1, 2)
        x = x + self.drop1(y)

        y = self.norm2(x)
        y = self.ffn(y)
        x = x + y
        return x

class HorizonAwareDecoder(nn.Module):
    """
    Horizon-aware decoder using horizon embeddings.
    """
    def __init__(self, d_model: int, max_pred_len: int = 192, dropout: float = 0.1):
        super().__init__()
        self.max_pred_len = max_pred_len
        self.horizon_emb = nn.Parameter(torch.randn(max_pred_len, d_model) * 0.02)

        self.context_proj = nn.Sequential(
            nn.LayerNorm(d_model * 2),
            nn.Linear(d_model * 2, d_model),
            nn.GELU(),
            nn.Dropout(dropout),
        )

        self.decoder = nn.Sequential(
            nn.LayerNorm(d_model),
            nn.Linear(d_model, d_model),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d_model, 1),
        )

    def forward(self, seq_repr, pred_len: int):
        if pred_len > self.max_pred_len:
            raise ValueError(f"pred_len={pred_len} exceeds max_pred_len={self.max_pred_len}")

        last = seq_repr[:, -1, :]
        mean = seq_repr.mean(dim=1)
        context = self.context_proj(torch.cat([last, mean], dim=-1))  # [B, D]

        hz = self.horizon_emb[:pred_len].unsqueeze(0).expand(seq_repr.size(0), -1, -1)  # [B, H, D]
        z = hz + context.unsqueeze(1)
        out = self.decoder(z).squeeze(-1)  # [B, H]
        return out

class AdaptiveWaveletMixer(nn.Module):
    """
    Phase 5 proposed model.
    """
    def __init__(
        self,
        input_dim: int,
        pred_len: int,
        d_model: int = 64,
        levels: int = 3,
        wavelet_family: str = "db4",
        num_backbone_blocks: int = 3,
        dropout: float = 0.1,
        filter_reg_lambda: float = 1e-4,
        gate_entropy_lambda: float = 1e-4,
    ):
        super().__init__()
        self.pred_len = pred_len
        self.input_proj = nn.Linear(input_dim, d_model)

        self.decomposer = AdaptiveWaveletDecomposer(
            d_model=d_model,
            wavelet_family=wavelet_family,
            levels=levels,
            filter_reg_lambda=filter_reg_lambda,
            gate_entropy_lambda=gate_entropy_lambda,
        )

        self.fusion = CrossScaleFusion(d_model=d_model, n_scales=levels + 2)

        self.backbone = nn.ModuleList([
            TemporalChannelMixerBlock(d_model=d_model, dropout=dropout, kernel_size=5)
            for _ in range(num_backbone_blocks)
        ])

        self.decoder = HorizonAwareDecoder(
            d_model=d_model,
            max_pred_len=max(192, pred_len),
            dropout=dropout,
        )

    def forward(self, x):
        # x: [B, T, C]
        raw_proj = self.input_proj(x)  # [B, T, D]

        decomp = self.decomposer(raw_proj)
        fused, fusion_attn = self.fusion(
            raw_proj=raw_proj,
            high_components=decomp["high_components"],
            final_low=decomp["final_low"],
        )

        z = fused
        for block in self.backbone:
            z = block(z)

        pred = self.decoder(z, pred_len=self.pred_len)

        aux = {
            "level_gates": decomp["gates"],
            "fusion_attn": fusion_attn,
        }
        return pred, aux

    def regularization_loss(self, aux):
        reg = self.decomposer.regularization_loss()
        reg = reg + self.decomposer.gate_entropy_loss(aux["level_gates"])
        return reg

# ------------------------------------------------------------
# Training / evaluation
# ------------------------------------------------------------

@torch.no_grad()
def evaluate_model(model, loader, target_mean: float, target_std: float):
    model.eval()
    criterion = nn.MSELoss()

    preds_all, trues_all = [], []
    scaled_loss_sum, n = 0.0, 0

    for batch in loader:
        x = batch["x_scaled"].to(DEVICE, non_blocking=True)
        y_scaled = batch["y_scaled"].to(DEVICE, non_blocking=True)
        y_raw = batch["y_raw"].cpu().numpy()

        with torch.cuda.amp.autocast(enabled=TRAIN_CFG["use_amp"]):
            pred_scaled, aux = model(x)
            loss = criterion(pred_scaled, y_scaled)

        pred_raw = pred_scaled.detach().cpu().numpy() * target_std + target_mean

        preds_all.append(pred_raw)
        trues_all.append(y_raw)
        scaled_loss_sum += loss.item() * x.size(0)
        n += x.size(0)

    preds_all = np.concatenate(preds_all, axis=0)
    trues_all = np.concatenate(trues_all, axis=0)

    metrics = compute_metrics(trues_all.reshape(-1), preds_all.reshape(-1))
    metrics["scaled_loss"] = float(scaled_loss_sum / max(n, 1))
    return metrics

def train_model(model, train_loader, val_loader, target_mean, target_std, max_epochs, patience):
    optimizer = torch.optim.AdamW(model.parameters(), lr=TRAIN_CFG["lr"], weight_decay=TRAIN_CFG["weight_decay"])
    scaler = torch.cuda.amp.GradScaler(enabled=TRAIN_CFG["use_amp"])
    criterion = nn.MSELoss()

    best_val = np.inf
    best_state = None
    patience_left = patience

    for epoch in range(1, max_epochs + 1):
        model.train()
        for batch in train_loader:
            x = batch["x_scaled"].to(DEVICE, non_blocking=True)
            y = batch["y_scaled"].to(DEVICE, non_blocking=True)

            optimizer.zero_grad(set_to_none=True)

            with torch.cuda.amp.autocast(enabled=TRAIN_CFG["use_amp"]):
                pred, aux = model(x)
                base_loss = criterion(pred, y)
                reg_loss = model.regularization_loss(aux)
                loss = base_loss + reg_loss

            scaler.scale(loss).backward()
            if TRAIN_CFG["grad_clip"] is not None:
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), TRAIN_CFG["grad_clip"])
            scaler.step(optimizer)
            scaler.update()

        val_metrics = evaluate_model(model, val_loader, target_mean, target_std)

        if val_metrics["rmse"] < best_val:
            best_val = val_metrics["rmse"]
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            patience_left = patience
        else:
            patience_left -= 1
            if patience_left <= 0:
                break

    model.load_state_dict(best_state)
    return model, best_val

def run_single_experiment(exp_cfg: Dict):
    set_seed(42)

    dataset_name = exp_cfg["dataset"]
    input_mode = exp_cfg["input_mode"]
    horizon_mode = exp_cfg["horizon_mode"]
    lookback_mode = exp_cfg["lookback_mode"]

    bundle = load_bundle(dataset_name, input_mode=input_mode)
    pred_len = resolve_pred_len(dataset_name, horizon_mode)
    seq_candidates = resolve_seq_candidates(dataset_name, lookback_mode)

    batch_size = int(exp_cfg.get("batch_size", DEFAULT_BATCH_SIZE[dataset_name]))
    full_epochs = int(exp_cfg.get("epochs", DEFAULT_EPOCHS[dataset_name]))

    target_idx = bundle["target_idx"]
    target_mean = float(bundle["scaler_mean"][target_idx])
    target_std = float(bundle["scaler_std"][target_idx])
    input_dim = int(bundle["values_scaled"].shape[1])

    # Search for best lookback
    chosen_seq = None
    best_search_val = np.inf

    for seq_len in seq_candidates:
        train_loader, val_loader, _ = make_loaders(bundle, seq_len, pred_len, batch_size)
        model = AdaptiveWaveletMixer(
            input_dim=input_dim,
            pred_len=pred_len,
            d_model=int(exp_cfg["d_model"]),
            levels=int(exp_cfg["levels"]),
            wavelet_family=exp_cfg["wavelet_family"],
            num_backbone_blocks=int(exp_cfg["num_backbone_blocks"]),
            dropout=float(exp_cfg["dropout"]),
            filter_reg_lambda=float(exp_cfg["filter_reg_lambda"]),
            gate_entropy_lambda=float(exp_cfg["gate_entropy_lambda"]),
        ).to(DEVICE)

        search_epochs = TRAIN_CFG["search_epochs"] if len(seq_candidates) > 1 else full_epochs
        search_patience = TRAIN_CFG["search_patience"] if len(seq_candidates) > 1 else TRAIN_CFG["full_patience"]

        model, search_val_rmse = train_model(
            model, train_loader, val_loader,
            target_mean=target_mean,
            target_std=target_std,
            max_epochs=search_epochs,
            patience=search_patience,
        )

        if search_val_rmse < best_search_val:
            best_search_val = search_val_rmse
            chosen_seq = seq_len

        del model
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    # Final train on chosen sequence length
    train_loader, val_loader, test_loader = make_loaders(bundle, chosen_seq, pred_len, batch_size)

    model = AdaptiveWaveletMixer(
        input_dim=input_dim,
        pred_len=pred_len,
        d_model=int(exp_cfg["d_model"]),
        levels=int(exp_cfg["levels"]),
        wavelet_family=exp_cfg["wavelet_family"],
        num_backbone_blocks=int(exp_cfg["num_backbone_blocks"]),
        dropout=float(exp_cfg["dropout"]),
        filter_reg_lambda=float(exp_cfg["filter_reg_lambda"]),
        gate_entropy_lambda=float(exp_cfg["gate_entropy_lambda"]),
    ).to(DEVICE)

    t0 = time.time()
    model, final_val_rmse = train_model(
        model, train_loader, val_loader,
        target_mean=target_mean,
        target_std=target_std,
        max_epochs=full_epochs,
        patience=TRAIN_CFG["full_patience"],
    )
    train_seconds = float(time.time() - t0)

    val_metrics = evaluate_model(model, val_loader, target_mean, target_std)
    test_metrics = evaluate_model(model, test_loader, target_mean, target_std)

    row = dict(exp_cfg)
    row.update({
        "model": "AdaptiveWaveletMixer",
        "family": "wavelet_adaptive",
        "chosen_seq_len": int(chosen_seq),
        "pred_len": int(pred_len),
        "input_dim": int(input_dim),
        "search_best_val_rmse": float(best_search_val),
        "val_mse": val_metrics["mse"],
        "val_mae": val_metrics["mae"],
        "val_rmse": val_metrics["rmse"],
        "test_mse": test_metrics["mse"],
        "test_mae": test_metrics["mae"],
        "test_rmse": test_metrics["rmse"],
        "train_seconds": train_seconds,
        "status": "ok",
    })

    del model
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return row

def run_experiment_grid(exp_df: pd.DataFrame, out_csv_name: str, overwrite: bool = False):
    out_path = PHASE5_TABLES / out_csv_name

    if out_path.exists() and not overwrite:
        existing = pd.read_csv(out_path)
        done = set(existing["exp_id"].astype(str).tolist())
    else:
        existing = pd.DataFrame()
        done = set()

    new_rows = []

    for i, row in exp_df.reset_index(drop=True).iterrows():
        exp_cfg = row.to_dict()
        exp_id = (
            f"{exp_cfg['dataset']}__{exp_cfg['input_mode']}__{exp_cfg['horizon_mode']}"
            f"__{exp_cfg['lookback_mode']}__{exp_cfg['wavelet_family']}"
            f"__L{exp_cfg['levels']}__dm{exp_cfg['d_model']}"
            f"__blk{exp_cfg['num_backbone_blocks']}"
        )
        exp_cfg["exp_id"] = exp_id

        if exp_id in done:
            continue

        print("=" * 120)
        print(f"Running {i+1}/{len(exp_df)} | {exp_id}")
        print("=" * 120)

        try:
            result = run_single_experiment(exp_cfg)
        except Exception as e:
            result = dict(exp_cfg)
            result["model"] = "AdaptiveWaveletMixer"
            result["family"] = "wavelet_adaptive"
            result["status"] = f"failed: {type(e).__name__}: {e}"

        new_rows.append(result)
        current = pd.concat([existing, pd.DataFrame(new_rows)], ignore_index=True)
        current.to_csv(out_path, index=False)

    final_df = pd.read_csv(out_path)
    return final_df
'''
runner_path.write_text(runner_code)
print("Wrote:", runner_path)

Wrote: /data/Sajjan_Singh/spml/wavelet_seq_project/scripts/phase5_adaptive_wavelet_model.py


In [3]:
# ============================================
# Cell 2: Import Phase 5 runner
# ============================================

import sys
sys.path.append(str(ROOT / "scripts"))

import phase5_adaptive_wavelet_model as p5

print("DEVICE:", p5.DEVICE)
print("ALL DATASETS:", p5.ALL_DATASETS)

DEVICE: cuda
ALL DATASETS: ['etth1', 'etth2', 'ettm1', 'ettm2', 'weather', 'electricity', 'traffic', 'exchange', 'ili']


In [4]:
# ============================================
# Cell 3: Build Phase 5 benchmark experiment grid
# ============================================

import pandas as pd

# Core Phase 5 benchmark:
# - all 9 datasets
# - multivariate
# - long horizon
# - searched lookback
# - proposed adaptive model
# - db4 analytic init
# - 3 levels
# - lightweight backbone

rows = []
for ds in p5.ALL_DATASETS:
    rows.append({
        "dataset": ds,
        "input_mode": "multivariate",
        "horizon_mode": "long",
        "lookback_mode": "searched",
        "wavelet_family": "Db4",
        "levels": 3,
        "d_model": 64,
        "num_backbone_blocks": 3,
        "dropout": 0.1,
        "filter_reg_lambda": 1e-4,
        "gate_entropy_lambda": 1e-4,
        "batch_size": p5.DEFAULT_BATCH_SIZE[ds],
        "epochs": p5.DEFAULT_EPOCHS[ds],
    })

phase5_exp_df = pd.DataFrame(rows)
display(phase5_exp_df)

,dataset,input_mode,horizon_mode,lookback_mode,wavelet_family,levels,d_model,num_backbone_blocks,dropout,filter_reg_lambda,gate_entropy_lambda,batch_size,epochs
0,etth1,multivariate,long,searched,Db4,3,64,3,0.1,0.0001,0.0001,256,30
1,etth2,multivariate,long,searched,Db4,3,64,3,0.1,0.0001,0.0001,256,30
2,ettm1,multivariate,long,searched,Db4,3,64,3,0.1,0.0001,0.0001,256,30
3,ettm2,multivariate,long,searched,Db4,3,64,3,0.1,0.0001,0.0001,256,30
4,weather,multivariate,long,searched,Db4,3,64,3,0.1,0.0001,0.0001,192,30
5,electricity,multivariate,long,searched,Db4,3,64,3,0.1,0.0001,0.0001,128,30
6,traffic,multivariate,long,searched,Db4,3,64,3,0.1,0.0001,0.0001,96,30
7,exchange,multivariate,long,searched,Db4,3,64,3,0.1,0.0001,0.0001,256,30
8,ili,multivariate,long,searched,Db4,3,64,3,0.1,0.0001,0.0001,64,35


In [5]:
# ============================================
# Cell 4: Optional quick pilot before full run
# ============================================

# Set RUN_FULL = True to run all 9 datasets.
RUN_FULL = True

if RUN_FULL:
    run_df = phase5_exp_df.copy()
else:
    # quick pilot on 3 datasets
    run_df = phase5_exp_df[phase5_exp_df["dataset"].isin(["etth1", "weather", "traffic"])].copy()

print("Experiments to run:", len(run_df))
display(run_df)

Experiments to run: 9


,dataset,input_mode,horizon_mode,lookback_mode,wavelet_family,levels,d_model,num_backbone_blocks,dropout,filter_reg_lambda,gate_entropy_lambda,batch_size,epochs
0,etth1,multivariate,long,searched,Db4,3,64,3,0.1,0.0001,0.0001,256,30
1,etth2,multivariate,long,searched,Db4,3,64,3,0.1,0.0001,0.0001,256,30
2,ettm1,multivariate,long,searched,Db4,3,64,3,0.1,0.0001,0.0001,256,30
3,ettm2,multivariate,long,searched,Db4,3,64,3,0.1,0.0001,0.0001,256,30
4,weather,multivariate,long,searched,Db4,3,64,3,0.1,0.0001,0.0001,192,30
5,electricity,multivariate,long,searched,Db4,3,64,3,0.1,0.0001,0.0001,128,30
6,traffic,multivariate,long,searched,Db4,3,64,3,0.1,0.0001,0.0001,96,30
7,exchange,multivariate,long,searched,Db4,3,64,3,0.1,0.0001,0.0001,256,30
8,ili,multivariate,long,searched,Db4,3,64,3,0.1,0.0001,0.0001,64,35


In [6]:
# ============================================
# Cell 5: Run Phase 5 adaptive model benchmark
# ============================================

phase5_results = p5.run_experiment_grid(
    exp_df=run_df,
    out_csv_name="phase5_adaptive_wavelet_mixer_metrics.csv",
    overwrite=False,
)

print("Rows:", len(phase5_results))
display(phase5_results.tail(10))

Running 1/9 | etth1__multivariate__long__searched__Db4__L3__dm64__blk3


/data/Sajjan_Singh/spml/wavelet_seq_project/scripts/phase5_adaptive_wavelet_model.py:567: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=TRAIN_CFG["use_amp"])
/data/Sajjan_Singh/spml/wavelet_seq_project/scripts/phase5_adaptive_wavelet_model.py:582: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=TRAIN_CFG["use_amp"]):
/data/Sajjan_Singh/spml/wavelet_seq_project/scripts/phase5_adaptive_wavelet_model.py:547: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=TRAIN_CFG["use_amp"]):


Running 2/9 | etth2__multivariate__long__searched__Db4__L3__dm64__blk3
Running 3/9 | ettm1__multivariate__long__searched__Db4__L3__dm64__blk3
Running 4/9 | ettm2__multivariate__long__searched__Db4__L3__dm64__blk3
Running 5/9 | weather__multivariate__long__searched__Db4__L3__dm64__blk3
Running 6/9 | electricity__multivariate__long__searched__Db4__L3__dm64__blk3
Running 7/9 | traffic__multivariate__long__searched__Db4__L3__dm64__blk3
Running 8/9 | exchange__multivariate__long__searched__Db4__L3__dm64__blk3
Running 9/9 | ili__multivariate__long__searched__Db4__L3__dm64__blk3
Rows: 9


,dataset,input_mode,horizon_mode,lookback_mode,wavelet_family,levels,d_model,num_backbone_blocks,dropout,filter_reg_lambda,...,input_dim,search_best_val_rmse,val_mse,val_mae,val_rmse,test_mse,test_mae,test_rmse,train_seconds,status
0,etth1,multivariate,long,searched,Db4,3,64,3,0.1,0.0001,...,7,3.379965,1.399415e+01,3.057286,3.740876,2.958256e+01,4.912198,5.438986,9.743361,ok
1,etth2,multivariate,long,searched,Db4,3,64,3,0.1,0.0001,...,7,6.812503,5.846906e+01,6.248178,7.646507,3.199892e+02,16.835471,17.888242,8.283108,ok
2,ettm1,multivariate,long,searched,Db4,3,64,3,0.1,0.0001,...,7,2.668290,8.050856e+00,2.246461,2.837403,1.835789e+01,3.719169,4.284610,35.507010,ok
3,ettm2,multivariate,long,searched,Db4,3,64,3,0.1,0.0001,...,7,4.929865,2.721885e+01,4.038539,5.217169,3.026552e+01,4.446908,5.501410,40.331134,ok
4,weather,multivariate,long,searched,Db4,3,64,3,0.1,0.0001,...,21,20.383592,6.259884e+02,16.706664,25.019759,6.195218e+02,19.383595,24.890195,42.102653,ok
5,electricity,multivariate,long,searched,Db4,3,64,3,0.1,0.0001,...,321,295.028198,7.349094e+04,189.998185,271.092123,1.219052e+05,262.232980,349.149250,90.188613,ok
6,traffic,multivariate,long,searched,Db4,3,64,3,0.1,0.0001,...,862,0.006781,4.258076e-05,0.004758,0.006525,9.932605e-05,0.006835,0.009966,249.106278,ok
7,exchange,multivariate,long,searched,Db4,3,64,3,0.1,0.0001,...,8,0.066400,5.552093e-03,0.065462,0.074512,7.696394e-03,0.071833,0.087729,7.587595,ok
8,ili,multivariate,long,searched,Db4,3,64,3,0.1,0.0001,...,7,83611.912630,3.324396e+09,43387.204955,57657.576567,2.377397e+11,420595.875368,487585.537890,20.093722,ok


In [7]:
# ============================================
# Cell 6: Summarize Phase 5 results
# ============================================

from pathlib import Path
import pandas as pd

PHASE5_DIR = ROOT / "results" / "tables" / "phase5"
phase5_csv = PHASE5_DIR / "phase5_adaptive_wavelet_mixer_metrics.csv"

phase5_df = pd.read_csv(phase5_csv)
print("Phase 5 rows:", len(phase5_df))

print("\nStatus counts:")
print(phase5_df["status"].astype(str).value_counts(dropna=False).to_string())

ok = phase5_df[phase5_df["status"].astype(str) == "ok"].copy()
print("\nUsable rows:", len(ok))

if len(ok) > 0:
    show_cols = [
        "dataset", "model", "family", "wavelet_family", "levels",
        "chosen_seq_len", "pred_len", "test_mse", "test_mae", "test_rmse",
        "train_seconds"
    ]
    display(ok[show_cols].sort_values(["dataset"]))

    summary = ok.groupby("model", as_index=False)[["test_mse", "test_mae", "test_rmse"]].mean()
    print("\nAverage metrics:")
    display(summary)

Phase 5 rows: 9

Status counts:
status
ok    9

Usable rows: 9


,dataset,model,family,wavelet_family,levels,chosen_seq_len,pred_len,test_mse,test_mae,test_rmse,train_seconds
5,electricity,AdaptiveWaveletMixer,wavelet_adaptive,Db4,3,96,96,1.219052e+05,262.232980,349.149250,90.188613
0,etth1,AdaptiveWaveletMixer,wavelet_adaptive,Db4,3,168,96,2.958256e+01,4.912198,5.438986,9.743361
1,etth2,AdaptiveWaveletMixer,wavelet_adaptive,Db4,3,96,96,3.199892e+02,16.835471,17.888242,8.283108
2,ettm1,AdaptiveWaveletMixer,wavelet_adaptive,Db4,3,336,96,1.835789e+01,3.719169,4.284610,35.507010
3,ettm2,AdaptiveWaveletMixer,wavelet_adaptive,Db4,3,96,96,3.026552e+01,4.446908,5.501410,40.331134
7,exchange,AdaptiveWaveletMixer,wavelet_adaptive,Db4,3,48,96,7.696394e-03,0.071833,0.087729,7.587595
8,ili,AdaptiveWaveletMixer,wavelet_adaptive,Db4,3,26,24,2.377397e+11,420595.875368,487585.537890,20.093722
6,traffic,AdaptiveWaveletMixer,wavelet_adaptive,Db4,3,336,96,9.932605e-05,0.006835,0.009966,249.106278
4,weather,AdaptiveWaveletMixer,wavelet_adaptive,Db4,3,96,96,6.195218e+02,19.383595,24.890195,42.102653



Average metrics:


,model,test_mse,test_mae,test_rmse
0,AdaptiveWaveletMixer,2.641553e+10,46767.498262,54221.42092


In [8]:
# ============================================
# Cell 7: Merge Phase 5 results into master CSV
# ============================================

import pandas as pd

MASTER_PATH = ROOT / "results" / "tables" / "master_all_models_metrics.csv"
PHASE5_PATH = ROOT / "results" / "tables" / "phase5" / "phase5_adaptive_wavelet_mixer_metrics.csv"

master = pd.read_csv(MASTER_PATH)
phase5_df = pd.read_csv(PHASE5_PATH)

phase5_ok = phase5_df[phase5_df["status"].astype(str) == "ok"].copy()

for col in master.columns:
    if col not in phase5_ok.columns:
        phase5_ok[col] = None

phase5_ok = phase5_ok[master.columns]
updated_master = pd.concat([master, phase5_ok], ignore_index=True)
updated_master = updated_master.sort_values(["dataset", "family", "model"]).reset_index(drop=True)

updated_master.to_csv(MASTER_PATH, index=False)

print("Updated master:", MASTER_PATH)
print("Rows:", len(updated_master))

best = (
    updated_master.sort_values(["dataset", "test_rmse", "test_mae", "test_mse"])
                  .groupby("dataset", as_index=False)
                  .first()[["dataset", "model", "family", "test_mse", "test_mae", "test_rmse"]]
)

display(best)

Updated master: /data/Sajjan_Singh/spml/wavelet_seq_project/results/tables/master_all_models_metrics.csv
Rows: 126


/tmp/ipykernel_2332903/1465535462.py:20: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  updated_master = pd.concat([master, phase5_ok], ignore_index=True)


,dataset,model,family,test_mse,test_mae,test_rmse
0,electricity,WPMixer,modern_tslib,0.218807,0.328796,0.467769
1,etth1,WPMixer,modern_tslib,0.055708,0.178231,0.236025
2,etth2,WPMixer,modern_tslib,0.132968,0.280923,0.364647
3,ettm1,WPMixer,modern_tslib,0.027799,0.125130,0.166729
4,ettm2,WPMixer,modern_tslib,0.063345,0.182120,0.251685
5,exchange,naive,classical,0.000795,0.021018,0.028205
6,ili,iTransformer,modern_tslib,0.628020,0.549220,0.792477
7,traffic,dlinear,neural_non_wavelet,0.000045,0.003913,0.006687
8,weather,WPMixer,modern_tslib,0.001172,0.026169,0.034238


In [9]:
# ============================================
# NEW FINAL CELL: Save Phase 5 checkpoints + raw-unit test predictions
# ============================================

from pathlib import Path
import gc
import numpy as np
import pandas as pd
import torch

ROOT = Path("/data/Sajjan_Singh/spml/wavelet_seq_project")
PHASE5_DIR = ROOT / "results" / "tables" / "phase5"
PHASE5_CSV = PHASE5_DIR / "phase5_adaptive_wavelet_mixer_metrics.csv"
PHASE5_REGISTRY = PHASE5_DIR / "phase5_prediction_registry.csv"
CKPT_DIR = ROOT / "results" / "checkpoints" / "phase5"
PRED_DIR = ROOT / "results" / "predictions" / "phase5"

CKPT_DIR.mkdir(parents=True, exist_ok=True)
PRED_DIR.mkdir(parents=True, exist_ok=True)

phase5_df = pd.read_csv(PHASE5_CSV)
ok = phase5_df[phase5_df["status"].astype(str) == "ok"].copy()

def row_to_cfg(row):
    return {
        "dataset": row["dataset"],
        "input_mode": row.get("input_mode", "multivariate"),
        "horizon_mode": row.get("horizon_mode", "long"),
        "lookback_mode": row.get("lookback_mode", "searched"),
        "wavelet_family": row.get("wavelet_family", "Db4"),
        "levels": int(row.get("levels", 3)),
        "d_model": int(row.get("d_model", 64)),
        "num_backbone_blocks": int(row.get("num_backbone_blocks", 3)),
        "dropout": float(row.get("dropout", 0.1)),
        "filter_reg_lambda": float(row.get("filter_reg_lambda", 1e-4)),
        "gate_entropy_lambda": float(row.get("gate_entropy_lambda", 1e-4)),
        "batch_size": int(row.get("batch_size", p5.DEFAULT_BATCH_SIZE[row["dataset"]])),
        "epochs": int(row.get("epochs", p5.DEFAULT_EPOCHS[row["dataset"]])),
    }

@torch.no_grad()
def save_phase5_preds(model, loader, target_mean, target_std, pred_path):
    model.eval()
    preds, trues = [], []
    for batch in loader:
        x = batch["x_scaled"].to(p5.DEVICE, non_blocking=True)
        y_raw = batch["y_raw"].cpu().numpy()
        with torch.cuda.amp.autocast(enabled=p5.TRAIN_CFG["use_amp"]):
            pred_scaled, aux = model(x)
        pred_raw = pred_scaled.detach().cpu().numpy() * target_std + target_mean
        preds.append(pred_raw)
        trues.append(y_raw)
    preds = np.concatenate(preds, axis=0)
    trues = np.concatenate(trues, axis=0)
    np.savez_compressed(pred_path, preds=preds, trues=trues)

registry_rows = []

for _, row in ok.iterrows():
    cfg = row_to_cfg(row)
    ds = cfg["dataset"]

    print("\n" + "=" * 100)
    print("PHASE5 SAVE:", ds)
    print("=" * 100)

    bundle = p5.load_bundle(ds, input_mode=cfg["input_mode"])
    pred_len = p5.resolve_pred_len(ds, cfg["horizon_mode"])

    if "chosen_seq_len" in row and pd.notna(row["chosen_seq_len"]):
        seq_len = int(row["chosen_seq_len"])
    else:
        seq_len = p5.resolve_seq_candidates(ds, cfg["lookback_mode"])[1]

    train_loader, val_loader, test_loader = p5.make_loaders(bundle, seq_len, pred_len, cfg["batch_size"])
    target_idx = bundle["target_idx"]
    target_mean = float(bundle["scaler_mean"][target_idx])
    target_std = float(bundle["scaler_std"][target_idx])

    model = p5.AdaptiveWaveletMixer(
        input_dim=int(bundle["values_scaled"].shape[1]),
        pred_len=pred_len,
        d_model=int(cfg["d_model"]),
        levels=int(cfg["levels"]),
        wavelet_family=cfg["wavelet_family"],
        num_backbone_blocks=int(cfg["num_backbone_blocks"]),
        dropout=float(cfg["dropout"]),
        filter_reg_lambda=float(cfg["filter_reg_lambda"]),
        gate_entropy_lambda=float(cfg["gate_entropy_lambda"]),
    ).to(p5.DEVICE)

    # retrain to recover best state for saving
    model, best_val = p5.train_model(
        model=model,
        train_loader=train_loader,
        val_loader=val_loader,
        target_mean=target_mean,
        target_std=target_std,
        max_epochs=int(cfg["epochs"]),
        patience=p5.TRAIN_CFG["full_patience"],
    )

    ckpt_path = CKPT_DIR / f"{ds}_adaptive_wavelet_mixer.pt"
    pred_path = PRED_DIR / f"{ds}_adaptive_wavelet_mixer_test_predictions.npz"

    torch.save({
        "dataset": ds,
        "cfg": cfg,
        "seq_len": seq_len,
        "pred_len": pred_len,
        "state_dict": model.state_dict(),
    }, ckpt_path)

    save_phase5_preds(model, test_loader, target_mean, target_std, pred_path)

    data = np.load(pred_path)
    preds = data["preds"]
    trues = data["trues"]

    registry_rows.append({
        "dataset": ds,
        "model": "AdaptiveWaveletMixer",
        "family": "wavelet_adaptive",
        "seq_len": seq_len,
        "pred_len": pred_len,
        "checkpoint": str(ckpt_path.relative_to(ROOT)),
        "prediction_file": str(pred_path.relative_to(ROOT)),
        "test_mse": float(np.mean((preds - trues) ** 2)),
        "test_mae": float(np.mean(np.abs(preds - trues))),
        "test_rmse": float(np.sqrt(np.mean((preds - trues) ** 2))),
    })

    del model
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

registry_df = pd.DataFrame(registry_rows)
registry_df.to_csv(PHASE5_REGISTRY, index=False)

print("\nSaved:", PHASE5_REGISTRY)
display(registry_df)


PHASE5 SAVE: etth1


/data/Sajjan_Singh/spml/wavelet_seq_project/scripts/phase5_adaptive_wavelet_model.py:567: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=TRAIN_CFG["use_amp"])
/data/Sajjan_Singh/spml/wavelet_seq_project/scripts/phase5_adaptive_wavelet_model.py:582: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=TRAIN_CFG["use_amp"]):
/data/Sajjan_Singh/spml/wavelet_seq_project/scripts/phase5_adaptive_wavelet_model.py:547: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=TRAIN_CFG["use_amp"]):
/tmp/ipykernel_2332903/3523548538.py:48: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  


PHASE5 SAVE: etth2

PHASE5 SAVE: ettm1

PHASE5 SAVE: ettm2

PHASE5 SAVE: weather

PHASE5 SAVE: electricity

PHASE5 SAVE: traffic

PHASE5 SAVE: exchange

PHASE5 SAVE: ili

Saved: /data/Sajjan_Singh/spml/wavelet_seq_project/results/tables/phase5/phase5_prediction_registry.csv


,dataset,model,family,seq_len,pred_len,checkpoint,prediction_file,test_mse,test_mae,test_rmse
0,etth1,AdaptiveWaveletMixer,wavelet_adaptive,168,96,results/checkpoints/phase5/etth1_adaptive_wave...,results/predictions/phase5/etth1_adaptive_wave...,1.863704e+01,3.742036,4.317063
1,etth2,AdaptiveWaveletMixer,wavelet_adaptive,96,96,results/checkpoints/phase5/etth2_adaptive_wave...,results/predictions/phase5/etth2_adaptive_wave...,2.615651e+02,15.101347,16.172974
2,ettm1,AdaptiveWaveletMixer,wavelet_adaptive,336,96,results/checkpoints/phase5/ettm1_adaptive_wave...,results/predictions/phase5/ettm1_adaptive_wave...,1.981942e+01,3.809999,4.451900
3,ettm2,AdaptiveWaveletMixer,wavelet_adaptive,96,96,results/checkpoints/phase5/ettm2_adaptive_wave...,results/predictions/phase5/ettm2_adaptive_wave...,4.504209e+01,5.646016,6.711340
4,weather,AdaptiveWaveletMixer,wavelet_adaptive,96,96,results/checkpoints/phase5/weather_adaptive_wa...,results/predictions/phase5/weather_adaptive_wa...,9.052887e+02,25.112688,30.088017
5,electricity,AdaptiveWaveletMixer,wavelet_adaptive,96,96,results/checkpoints/phase5/electricity_adaptiv...,results/predictions/phase5/electricity_adaptiv...,1.207024e+05,262.943115,347.422485
6,traffic,AdaptiveWaveletMixer,wavelet_adaptive,336,96,results/checkpoints/phase5/traffic_adaptive_wa...,results/predictions/phase5/traffic_adaptive_wa...,9.472539e-05,0.006795,0.009733
7,exchange,AdaptiveWaveletMixer,wavelet_adaptive,48,96,results/checkpoints/phase5/exchange_adaptive_w...,results/predictions/phase5/exchange_adaptive_w...,8.479564e-03,0.076140,0.092085
8,ili,AdaptiveWaveletMixer,wavelet_adaptive,26,24,results/checkpoints/phase5/ili_adaptive_wavele...,results/predictions/phase5/ili_adaptive_wavele...,2.189168e+11,404970.343750,467885.500000


In [10]:
# ============================================
# New Cell 9: Save Phase 5 checkpoints and raw prediction files
# ============================================

from pathlib import Path
import gc
import numpy as np
import pandas as pd
import torch

ROOT = Path("/data/Sajjan_Singh/spml/wavelet_seq_project")
PHASE5_DIR = ROOT / "results" / "tables" / "phase5"
PHASE5_CSV = PHASE5_DIR / "phase5_adaptive_wavelet_mixer_metrics.csv"

CKPT_DIR = ROOT / "results" / "checkpoints" / "phase5"
PRED_DIR = ROOT / "results" / "predictions" / "phase5"
CKPT_DIR.mkdir(parents=True, exist_ok=True)
PRED_DIR.mkdir(parents=True, exist_ok=True)

phase5_df = pd.read_csv(PHASE5_CSV)
ok = phase5_df[phase5_df["status"].astype(str) == "ok"].copy()

def row_to_cfg(row):
    return {
        "dataset": row["dataset"],
        "input_mode": row.get("input_mode", "multivariate"),
        "horizon_mode": row.get("horizon_mode", "long"),
        "lookback_mode": row.get("lookback_mode", "searched"),
        "wavelet_family": row.get("wavelet_family", "Db4"),
        "levels": int(row.get("levels", 3)),
        "d_model": int(row.get("d_model", 64)),
        "num_backbone_blocks": int(row.get("num_backbone_blocks", 3)),
        "dropout": float(row.get("dropout", 0.1)),
        "filter_reg_lambda": float(row.get("filter_reg_lambda", 1e-4)),
        "gate_entropy_lambda": float(row.get("gate_entropy_lambda", 1e-4)),
        "batch_size": int(row.get("batch_size", p5.DEFAULT_BATCH_SIZE[row["dataset"]])),
        "epochs": int(row.get("epochs", p5.DEFAULT_EPOCHS[row["dataset"]])),
    }

@torch.no_grad()
def save_phase5_preds(model, loader, target_mean, target_std, pred_path):
    model.eval()
    preds, trues = [], []
    for batch in loader:
        x = batch["x_scaled"].to(p5.DEVICE, non_blocking=True)
        y_raw = batch["y_raw"].cpu().numpy()
        with torch.cuda.amp.autocast(enabled=p5.TRAIN_CFG["use_amp"]):
            pred_scaled, aux = model(x)
        pred_raw = pred_scaled.detach().cpu().numpy() * target_std + target_mean
        preds.append(pred_raw)
        trues.append(y_raw)
    preds = np.concatenate(preds, axis=0)
    trues = np.concatenate(trues, axis=0)
    np.savez_compressed(pred_path, preds=preds, trues=trues)

registry_rows = []

for _, row in ok.iterrows():
    cfg = row_to_cfg(row)
    ds = cfg["dataset"]

    print("\n" + "=" * 100)
    print("SAVING PHASE5 MODEL:", ds)
    print("=" * 100)

    bundle = p5.load_bundle(ds, input_mode=cfg["input_mode"])
    pred_len = p5.resolve_pred_len(ds, cfg["horizon_mode"])

    if "chosen_seq_len" in row and pd.notna(row["chosen_seq_len"]):
        seq_len = int(row["chosen_seq_len"])
    else:
        seq_len = p5.resolve_seq_candidates(ds, cfg["lookback_mode"])[1]

    train_loader, val_loader, test_loader = p5.make_loaders(bundle, seq_len, pred_len, cfg["batch_size"])
    target_idx = bundle["target_idx"]
    target_mean = float(bundle["scaler_mean"][target_idx])
    target_std = float(bundle["scaler_std"][target_idx])

    model = p5.AdaptiveWaveletMixer(
        input_dim=int(bundle["values_scaled"].shape[1]),
        pred_len=pred_len,
        d_model=int(cfg["d_model"]),
        levels=int(cfg["levels"]),
        wavelet_family=cfg["wavelet_family"],
        num_backbone_blocks=int(cfg["num_backbone_blocks"]),
        dropout=float(cfg["dropout"]),
        filter_reg_lambda=float(cfg["filter_reg_lambda"]),
        gate_entropy_lambda=float(cfg["gate_entropy_lambda"]),
    ).to(p5.DEVICE)

    model, best_val = p5.train_model(
        model=model,
        train_loader=train_loader,
        val_loader=val_loader,
        target_mean=target_mean,
        target_std=target_std,
        max_epochs=int(cfg["epochs"]),
        patience=p5.TRAIN_CFG["full_patience"],
    )

    ckpt_path = CKPT_DIR / f"{ds}_adaptive_wavelet_mixer.pt"
    pred_path = PRED_DIR / f"{ds}_adaptive_wavelet_mixer_test_predictions.npz"

    torch.save({
        "dataset": ds,
        "cfg": cfg,
        "seq_len": seq_len,
        "pred_len": pred_len,
        "state_dict": model.state_dict(),
    }, ckpt_path)

    save_phase5_preds(model, test_loader, target_mean, target_std, pred_path)

    arr = np.load(pred_path)
    preds = arr["preds"]
    trues = arr["trues"]

    registry_rows.append({
        "dataset": ds,
        "model": "AdaptiveWaveletMixer",
        "family": "wavelet_adaptive",
        "seq_len": seq_len,
        "pred_len": pred_len,
        "checkpoint": str(ckpt_path.relative_to(ROOT)),
        "prediction_file": str(pred_path.relative_to(ROOT)),
        "test_mse": float(np.mean((preds - trues) ** 2)),
        "test_mae": float(np.mean(np.abs(preds - trues))),
        "test_rmse": float(np.sqrt(np.mean((preds - trues) ** 2))),
    })

    del model
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

registry_df = pd.DataFrame(registry_rows)
registry_path = PHASE5_DIR / "phase5_prediction_registry.csv"
registry_df.to_csv(registry_path, index=False)

print("\nSaved:", registry_path)
display(registry_df)


SAVING PHASE5 MODEL: etth1


/data/Sajjan_Singh/spml/wavelet_seq_project/scripts/phase5_adaptive_wavelet_model.py:567: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=TRAIN_CFG["use_amp"])
/data/Sajjan_Singh/spml/wavelet_seq_project/scripts/phase5_adaptive_wavelet_model.py:582: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=TRAIN_CFG["use_amp"]):
/data/Sajjan_Singh/spml/wavelet_seq_project/scripts/phase5_adaptive_wavelet_model.py:547: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=TRAIN_CFG["use_amp"]):
/tmp/ipykernel_2332903/2874091889.py:47: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  


SAVING PHASE5 MODEL: etth2

SAVING PHASE5 MODEL: ettm1

SAVING PHASE5 MODEL: ettm2

SAVING PHASE5 MODEL: weather

SAVING PHASE5 MODEL: electricity

SAVING PHASE5 MODEL: traffic

SAVING PHASE5 MODEL: exchange

SAVING PHASE5 MODEL: ili

Saved: /data/Sajjan_Singh/spml/wavelet_seq_project/results/tables/phase5/phase5_prediction_registry.csv


,dataset,model,family,seq_len,pred_len,checkpoint,prediction_file,test_mse,test_mae,test_rmse
0,etth1,AdaptiveWaveletMixer,wavelet_adaptive,168,96,results/checkpoints/phase5/etth1_adaptive_wave...,results/predictions/phase5/etth1_adaptive_wave...,2.560059e+01,4.461848,5.059703
1,etth2,AdaptiveWaveletMixer,wavelet_adaptive,96,96,results/checkpoints/phase5/etth2_adaptive_wave...,results/predictions/phase5/etth2_adaptive_wave...,3.058757e+02,16.403313,17.489302
2,ettm1,AdaptiveWaveletMixer,wavelet_adaptive,336,96,results/checkpoints/phase5/ettm1_adaptive_wave...,results/predictions/phase5/ettm1_adaptive_wave...,9.919249e+00,2.488200,3.149484
3,ettm2,AdaptiveWaveletMixer,wavelet_adaptive,96,96,results/checkpoints/phase5/ettm2_adaptive_wave...,results/predictions/phase5/ettm2_adaptive_wave...,3.912944e+01,5.135308,6.255352
4,weather,AdaptiveWaveletMixer,wavelet_adaptive,96,96,results/checkpoints/phase5/weather_adaptive_wa...,results/predictions/phase5/weather_adaptive_wa...,4.039550e+02,14.299186,20.098633
5,electricity,AdaptiveWaveletMixer,wavelet_adaptive,96,96,results/checkpoints/phase5/electricity_adaptiv...,results/predictions/phase5/electricity_adaptiv...,1.335647e+05,274.131592,365.465088
6,traffic,AdaptiveWaveletMixer,wavelet_adaptive,336,96,results/checkpoints/phase5/traffic_adaptive_wa...,results/predictions/phase5/traffic_adaptive_wa...,1.043056e-04,0.006981,0.010213
7,exchange,AdaptiveWaveletMixer,wavelet_adaptive,48,96,results/checkpoints/phase5/exchange_adaptive_w...,results/predictions/phase5/exchange_adaptive_w...,6.605345e-03,0.067029,0.081273
8,ili,AdaptiveWaveletMixer,wavelet_adaptive,26,24,results/checkpoints/phase5/ili_adaptive_wavele...,results/predictions/phase5/ili_adaptive_wavele...,2.331582e+11,411899.218750,482864.562500


In [11]:
# ============================================
# New Cell 10: Compact architecture ablation for Phase 5
# ============================================

import pandas as pd

REP_DATASETS = ["etth1", "weather", "traffic"]

rows = []

# A) levels
for ds in REP_DATASETS:
    for levels in [2, 3, 4]:
        rows.append({
            "ablation_name": "levels",
            "dataset": ds,
            "input_mode": "multivariate",
            "horizon_mode": "long",
            "lookback_mode": "searched",
            "wavelet_family": "Db4",
            "levels": levels,
            "d_model": 64,
            "num_backbone_blocks": 3,
            "dropout": 0.1,
            "filter_reg_lambda": 1e-4,
            "gate_entropy_lambda": 1e-4,
            "batch_size": p5.DEFAULT_BATCH_SIZE[ds],
            "epochs": p5.DEFAULT_EPOCHS[ds],
        })

# B) wavelet family
for ds in REP_DATASETS:
    for fam in ["Db2", "Db4", "Symlet"]:
        rows.append({
            "ablation_name": "family",
            "dataset": ds,
            "input_mode": "multivariate",
            "horizon_mode": "long",
            "lookback_mode": "searched",
            "wavelet_family": fam,
            "levels": 3,
            "d_model": 64,
            "num_backbone_blocks": 3,
            "dropout": 0.1,
            "filter_reg_lambda": 1e-4,
            "gate_entropy_lambda": 1e-4,
            "batch_size": p5.DEFAULT_BATCH_SIZE[ds],
            "epochs": p5.DEFAULT_EPOCHS[ds],
        })

# C) model width
for ds in REP_DATASETS:
    for d_model in [64, 128]:
        rows.append({
            "ablation_name": "width",
            "dataset": ds,
            "input_mode": "multivariate",
            "horizon_mode": "long",
            "lookback_mode": "searched",
            "wavelet_family": "Db4",
            "levels": 3,
            "d_model": d_model,
            "num_backbone_blocks": 3,
            "dropout": 0.1,
            "filter_reg_lambda": 1e-4,
            "gate_entropy_lambda": 1e-4,
            "batch_size": p5.DEFAULT_BATCH_SIZE[ds],
            "epochs": p5.DEFAULT_EPOCHS[ds],
        })

# D) backbone depth
for ds in REP_DATASETS:
    for blocks in [2, 3, 4]:
        rows.append({
            "ablation_name": "depth",
            "dataset": ds,
            "input_mode": "multivariate",
            "horizon_mode": "long",
            "lookback_mode": "searched",
            "wavelet_family": "Db4",
            "levels": 3,
            "d_model": 64,
            "num_backbone_blocks": blocks,
            "dropout": 0.1,
            "filter_reg_lambda": 1e-4,
            "gate_entropy_lambda": 1e-4,
            "batch_size": p5.DEFAULT_BATCH_SIZE[ds],
            "epochs": p5.DEFAULT_EPOCHS[ds],
        })

phase5_arch_df = pd.DataFrame(rows)
display(phase5_arch_df.head(20))
print("Total architecture-ablation experiments:", len(phase5_arch_df))

,ablation_name,dataset,input_mode,horizon_mode,lookback_mode,wavelet_family,levels,d_model,num_backbone_blocks,dropout,filter_reg_lambda,gate_entropy_lambda,batch_size,epochs
0,levels,etth1,multivariate,long,searched,Db4,2,64,3,0.1,0.0001,0.0001,256,30
1,levels,etth1,multivariate,long,searched,Db4,3,64,3,0.1,0.0001,0.0001,256,30
2,levels,etth1,multivariate,long,searched,Db4,4,64,3,0.1,0.0001,0.0001,256,30
3,levels,weather,multivariate,long,searched,Db4,2,64,3,0.1,0.0001,0.0001,192,30
4,levels,weather,multivariate,long,searched,Db4,3,64,3,0.1,0.0001,0.0001,192,30
5,levels,weather,multivariate,long,searched,Db4,4,64,3,0.1,0.0001,0.0001,192,30
6,levels,traffic,multivariate,long,searched,Db4,2,64,3,0.1,0.0001,0.0001,96,30
7,levels,traffic,multivariate,long,searched,Db4,3,64,3,0.1,0.0001,0.0001,96,30
8,levels,traffic,multivariate,long,searched,Db4,4,64,3,0.1,0.0001,0.0001,96,30
9,family,etth1,multivariate,long,searched,Db2,3,64,3,0.1,0.0001,0.0001,256,30


Total architecture-ablation experiments: 33


In [12]:
phase5_arch_results = p5.run_experiment_grid(
    exp_df=phase5_arch_df,
    out_csv_name="phase5_architecture_ablation_metrics.csv",
    overwrite=False,
)

print("Rows:", len(phase5_arch_results))
display(phase5_arch_results.tail(10))

Running 1/33 | etth1__multivariate__long__searched__Db4__L2__dm64__blk3


/data/Sajjan_Singh/spml/wavelet_seq_project/scripts/phase5_adaptive_wavelet_model.py:567: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=TRAIN_CFG["use_amp"])
/data/Sajjan_Singh/spml/wavelet_seq_project/scripts/phase5_adaptive_wavelet_model.py:582: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=TRAIN_CFG["use_amp"]):
/data/Sajjan_Singh/spml/wavelet_seq_project/scripts/phase5_adaptive_wavelet_model.py:547: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=TRAIN_CFG["use_amp"]):


Running 2/33 | etth1__multivariate__long__searched__Db4__L3__dm64__blk3
Running 3/33 | etth1__multivariate__long__searched__Db4__L4__dm64__blk3
Running 4/33 | weather__multivariate__long__searched__Db4__L2__dm64__blk3
Running 5/33 | weather__multivariate__long__searched__Db4__L3__dm64__blk3
Running 6/33 | weather__multivariate__long__searched__Db4__L4__dm64__blk3
Running 7/33 | traffic__multivariate__long__searched__Db4__L2__dm64__blk3
Running 8/33 | traffic__multivariate__long__searched__Db4__L3__dm64__blk3
Running 9/33 | traffic__multivariate__long__searched__Db4__L4__dm64__blk3
Running 10/33 | etth1__multivariate__long__searched__Db2__L3__dm64__blk3
Running 11/33 | etth1__multivariate__long__searched__Db4__L3__dm64__blk3
Running 12/33 | etth1__multivariate__long__searched__Symlet__L3__dm64__blk3
Running 13/33 | weather__multivariate__long__searched__Db2__L3__dm64__blk3
Running 14/33 | weather__multivariate__long__searched__Db4__L3__dm64__blk3
Running 15/33 | weather__multivariate__l

,ablation_name,dataset,input_mode,horizon_mode,lookback_mode,wavelet_family,levels,d_model,num_backbone_blocks,dropout,...,input_dim,search_best_val_rmse,val_mse,val_mae,val_rmse,test_mse,test_mae,test_rmse,train_seconds,status
23,width,traffic,multivariate,long,searched,Db4,3,128,3,0.1,...,862,0.006284,0.000031,0.003874,0.005533,0.000077,0.005601,0.008787,152.158874,ok
24,depth,etth1,multivariate,long,searched,Db4,3,64,2,0.1,...,7,3.424782,11.278936,2.711972,3.358413,27.373446,4.726367,5.231964,8.253814,ok
25,depth,etth1,multivariate,long,searched,Db4,3,64,3,0.1,...,7,3.379965,13.994154,3.057286,3.740876,29.582564,4.912198,5.438986,8.826449,ok
26,depth,etth1,multivariate,long,searched,Db4,3,64,4,0.1,...,7,3.644376,12.473704,2.921515,3.531813,26.849357,4.629353,5.181637,9.541155,ok
27,depth,weather,multivariate,long,searched,Db4,3,64,2,0.1,...,21,19.923722,363.011856,14.007330,19.052870,398.328605,14.449262,19.958171,170.406307,ok
28,depth,weather,multivariate,long,searched,Db4,3,64,3,0.1,...,21,20.383592,625.988358,16.706664,25.019759,619.521818,19.383595,24.890195,48.991362,ok
29,depth,weather,multivariate,long,searched,Db4,3,64,4,0.1,...,21,21.807149,10565.305813,25.278230,102.787673,666.742349,20.402070,25.821355,50.120693,ok
30,depth,traffic,multivariate,long,searched,Db4,3,64,2,0.1,...,862,0.006574,0.000035,0.004343,0.005954,0.000088,0.006285,0.009403,102.639376,ok
31,depth,traffic,multivariate,long,searched,Db4,3,64,3,0.1,...,862,0.006781,0.000043,0.004758,0.006525,0.000099,0.006835,0.009966,199.482447,ok
32,depth,traffic,multivariate,long,searched,Db4,3,64,4,0.1,...,862,0.006775,0.000034,0.004194,0.005835,0.000076,0.005787,0.008742,150.053496,ok


In [13]:
# ============================================
# New Cell 12: Summarize Phase 5 architecture ablation
# ============================================

PHASE5_ARCH_PATH = ROOT / "results" / "tables" / "phase5" / "phase5_architecture_ablation_metrics.csv"
arch_df = pd.read_csv(PHASE5_ARCH_PATH)
arch_ok = arch_df[arch_df["status"].astype(str) == "ok"].copy()

print("Usable architecture-ablation rows:", len(arch_ok))

for ablation_name in sorted(arch_ok["ablation_name"].unique()):
    sub = arch_ok[arch_ok["ablation_name"] == ablation_name].copy()

    group_cols = {
        "levels": ["levels"],
        "family": ["wavelet_family"],
        "width": ["d_model"],
        "depth": ["num_backbone_blocks"],
    }[ablation_name]

    avg = (
        sub.groupby(group_cols, as_index=False)[["test_mse", "test_mae", "test_rmse"]]
           .mean()
           .sort_values("test_rmse")
           .reset_index(drop=True)
    )

    out_path = ROOT / "results" / "tables" / "phase5" / f"summary_phase5_{ablation_name}.csv"
    avg.to_csv(out_path, index=False)

    print("\nAblation:", ablation_name)
    print("Saved:", out_path)
    display(avg)

Usable architecture-ablation rows: 33

Ablation: depth
Saved: /data/Sajjan_Singh/spml/wavelet_seq_project/results/tables/phase5/summary_phase5_depth.csv


,num_backbone_blocks,test_mse,test_mae,test_rmse
0,2,141.900713,6.393971,8.399846
1,3,216.368160,8.100876,10.113049
2,4,231.197261,8.345737,10.337244



Ablation: family
Saved: /data/Sajjan_Singh/spml/wavelet_seq_project/results/tables/phase5/summary_phase5_family.csv


,wavelet_family,test_mse,test_mae,test_rmse
0,Db4,216.368160,8.100876,10.113049
1,Db2,278.490758,9.013794,11.279488
2,Symlet,2128.576215,7.851454,28.374833



Ablation: levels
Saved: /data/Sajjan_Singh/spml/wavelet_seq_project/results/tables/phase5/summary_phase5_levels.csv


,levels,test_mse,test_mae,test_rmse
0,2,110.834669,6.300967,7.792918
1,4,166.720285,7.245109,9.285625
2,3,216.368160,8.100876,10.113049



Ablation: width
Saved: /data/Sajjan_Singh/spml/wavelet_seq_project/results/tables/phase5/summary_phase5_width.csv


,d_model,test_mse,test_mae,test_rmse
0,128,121.197029,5.823263,7.699728
1,64,216.368160,8.100876,10.113049


In [14]:
# ============================================
# NEW FINAL CELL: Save Phase 5 checkpoints + raw test predictions
# ============================================

from pathlib import Path
import gc
import numpy as np
import pandas as pd
import torch

ROOT = Path("/data/Sajjan_Singh/spml/wavelet_seq_project")
PHASE5_DIR = ROOT / "results" / "tables" / "phase5"
PHASE5_CSV = PHASE5_DIR / "phase5_adaptive_wavelet_mixer_metrics.csv"

CKPT_DIR = ROOT / "results" / "checkpoints" / "phase5"
PRED_DIR = ROOT / "results" / "predictions" / "phase5"

CKPT_DIR.mkdir(parents=True, exist_ok=True)
PRED_DIR.mkdir(parents=True, exist_ok=True)

phase5_df = pd.read_csv(PHASE5_CSV)
ok = phase5_df[phase5_df["status"].astype(str) == "ok"].copy()

def row_to_cfg(row):
    return {
        "dataset": row["dataset"],
        "input_mode": row.get("input_mode", "multivariate"),
        "horizon_mode": row.get("horizon_mode", "long"),
        "lookback_mode": row.get("lookback_mode", "searched"),
        "wavelet_family": row.get("wavelet_family", "Db4"),
        "levels": int(row.get("levels", 3)),
        "d_model": int(row.get("d_model", 64)),
        "num_backbone_blocks": int(row.get("num_backbone_blocks", 3)),
        "dropout": float(row.get("dropout", 0.1)),
        "filter_reg_lambda": float(row.get("filter_reg_lambda", 1e-4)),
        "gate_entropy_lambda": float(row.get("gate_entropy_lambda", 1e-4)),
        "batch_size": int(row.get("batch_size", p5.DEFAULT_BATCH_SIZE[row["dataset"]])),
        "epochs": int(row.get("epochs", p5.DEFAULT_EPOCHS[row["dataset"]])),
    }

@torch.no_grad()
def save_phase5_preds(model, loader, target_mean, target_std, pred_path):
    model.eval()
    preds, trues = [], []
    for batch in loader:
        x = batch["x_scaled"].to(p5.DEVICE, non_blocking=True)
        y_raw = batch["y_raw"].cpu().numpy()
        with torch.cuda.amp.autocast(enabled=p5.TRAIN_CFG["use_amp"]):
            pred_scaled, aux = model(x)
        pred_raw = pred_scaled.detach().cpu().numpy() * target_std + target_mean
        preds.append(pred_raw)
        trues.append(y_raw)
    preds = np.concatenate(preds, axis=0)
    trues = np.concatenate(trues, axis=0)
    np.savez_compressed(pred_path, preds=preds, trues=trues)

registry_rows = []

for _, row in ok.iterrows():
    cfg = row_to_cfg(row)
    ds = cfg["dataset"]

    print("\n" + "=" * 100)
    print("SAVING PHASE5 MODEL:", ds)
    print("=" * 100)

    bundle = p5.load_bundle(ds, input_mode=cfg["input_mode"])
    pred_len = p5.resolve_pred_len(ds, cfg["horizon_mode"])

    if "chosen_seq_len" in row and pd.notna(row["chosen_seq_len"]):
        seq_len = int(row["chosen_seq_len"])
    else:
        seq_len = p5.resolve_seq_candidates(ds, cfg["lookback_mode"])[1]

    train_loader, val_loader, test_loader = p5.make_loaders(bundle, seq_len, pred_len, cfg["batch_size"])
    target_idx = bundle["target_idx"]
    target_mean = float(bundle["scaler_mean"][target_idx])
    target_std = float(bundle["scaler_std"][target_idx])

    model = p5.AdaptiveWaveletMixer(
        input_dim=int(bundle["values_scaled"].shape[1]),
        pred_len=pred_len,
        d_model=int(cfg["d_model"]),
        levels=int(cfg["levels"]),
        wavelet_family=cfg["wavelet_family"],
        num_backbone_blocks=int(cfg["num_backbone_blocks"]),
        dropout=float(cfg["dropout"]),
        filter_reg_lambda=float(cfg["filter_reg_lambda"]),
        gate_entropy_lambda=float(cfg["gate_entropy_lambda"]),
    ).to(p5.DEVICE)

    model, best_val = p5.train_model(
        model=model,
        train_loader=train_loader,
        val_loader=val_loader,
        target_mean=target_mean,
        target_std=target_std,
        max_epochs=int(cfg["epochs"]),
        patience=p5.TRAIN_CFG["full_patience"],
    )

    ckpt_path = CKPT_DIR / f"{ds}_adaptive_wavelet_mixer.pt"
    pred_path = PRED_DIR / f"{ds}_adaptive_wavelet_mixer_test_predictions.npz"

    torch.save({
        "dataset": ds,
        "cfg": cfg,
        "seq_len": seq_len,
        "pred_len": pred_len,
        "state_dict": model.state_dict(),
    }, ckpt_path)

    save_phase5_preds(model, test_loader, target_mean, target_std, pred_path)

    arr = np.load(pred_path)
    preds = arr["preds"]
    trues = arr["trues"]

    registry_rows.append({
        "dataset": ds,
        "model": "AdaptiveWaveletMixer",
        "family": "wavelet_adaptive",
        "seq_len": seq_len,
        "pred_len": pred_len,
        "checkpoint": str(ckpt_path.relative_to(ROOT)),
        "prediction_file": str(pred_path.relative_to(ROOT)),
        "test_mse": float(np.mean((preds - trues) ** 2)),
        "test_mae": float(np.mean(np.abs(preds - trues))),
        "test_rmse": float(np.sqrt(np.mean((preds - trues) ** 2))),
    })

    del model
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

registry_df = pd.DataFrame(registry_rows)
registry_path = PHASE5_DIR / "phase5_prediction_registry.csv"
registry_df.to_csv(registry_path, index=False)

print("\nSaved:", registry_path)
display(registry_df)


SAVING PHASE5 MODEL: etth1


/data/Sajjan_Singh/spml/wavelet_seq_project/scripts/phase5_adaptive_wavelet_model.py:567: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=TRAIN_CFG["use_amp"])
/data/Sajjan_Singh/spml/wavelet_seq_project/scripts/phase5_adaptive_wavelet_model.py:582: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=TRAIN_CFG["use_amp"]):
/data/Sajjan_Singh/spml/wavelet_seq_project/scripts/phase5_adaptive_wavelet_model.py:547: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=TRAIN_CFG["use_amp"]):
/tmp/ipykernel_2332903/3034502092.py:48: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  


SAVING PHASE5 MODEL: etth2

SAVING PHASE5 MODEL: ettm1

SAVING PHASE5 MODEL: ettm2

SAVING PHASE5 MODEL: weather

SAVING PHASE5 MODEL: electricity

SAVING PHASE5 MODEL: traffic

SAVING PHASE5 MODEL: exchange

SAVING PHASE5 MODEL: ili

Saved: /data/Sajjan_Singh/spml/wavelet_seq_project/results/tables/phase5/phase5_prediction_registry.csv


,dataset,model,family,seq_len,pred_len,checkpoint,prediction_file,test_mse,test_mae,test_rmse
0,etth1,AdaptiveWaveletMixer,wavelet_adaptive,168,96,results/checkpoints/phase5/etth1_adaptive_wave...,results/predictions/phase5/etth1_adaptive_wave...,3.079176e+01,5.021783,5.549033
1,etth2,AdaptiveWaveletMixer,wavelet_adaptive,96,96,results/checkpoints/phase5/etth2_adaptive_wave...,results/predictions/phase5/etth2_adaptive_wave...,3.398574e+02,17.427193,18.435221
2,ettm1,AdaptiveWaveletMixer,wavelet_adaptive,336,96,results/checkpoints/phase5/ettm1_adaptive_wave...,results/predictions/phase5/ettm1_adaptive_wave...,2.011030e+01,3.948575,4.484451
3,ettm2,AdaptiveWaveletMixer,wavelet_adaptive,96,96,results/checkpoints/phase5/ettm2_adaptive_wave...,results/predictions/phase5/ettm2_adaptive_wave...,3.162462e+01,4.699381,5.623577
4,weather,AdaptiveWaveletMixer,wavelet_adaptive,96,96,results/checkpoints/phase5/weather_adaptive_wa...,results/predictions/phase5/weather_adaptive_wa...,7.039667e+02,20.614586,26.532372
5,electricity,AdaptiveWaveletMixer,wavelet_adaptive,96,96,results/checkpoints/phase5/electricity_adaptiv...,results/predictions/phase5/electricity_adaptiv...,1.528667e+05,290.227356,390.981689
6,traffic,AdaptiveWaveletMixer,wavelet_adaptive,336,96,results/checkpoints/phase5/traffic_adaptive_wa...,results/predictions/phase5/traffic_adaptive_wa...,9.273746e-05,0.006621,0.009630
7,exchange,AdaptiveWaveletMixer,wavelet_adaptive,48,96,results/checkpoints/phase5/exchange_adaptive_w...,results/predictions/phase5/exchange_adaptive_w...,6.043110e-03,0.064121,0.077737
8,ili,AdaptiveWaveletMixer,wavelet_adaptive,26,24,results/checkpoints/phase5/ili_adaptive_wavele...,results/predictions/phase5/ili_adaptive_wavele...,2.161578e+11,398963.906250,464927.750000


In [2]:
# ============================================
# Phase 5 setup: import p5 safely
# ============================================

from pathlib import Path
import sys
import importlib

ROOT = Path("/data/Sajjan_Singh/spml/wavelet_seq_project")
SCRIPTS_DIR = ROOT / "scripts"
RUNNER_PATH = SCRIPTS_DIR / "phase5_adaptive_wavelet_model.py"

print("ROOT:", ROOT)
print("RUNNER_PATH:", RUNNER_PATH)

if not RUNNER_PATH.exists():
    raise FileNotFoundError(
        f"Missing runner file: {RUNNER_PATH}\n"
        "Please first run the earlier cell that writes phase5_adaptive_wavelet_model.py"
    )

if str(SCRIPTS_DIR) not in sys.path:
    sys.path.append(str(SCRIPTS_DIR))

import phase5_adaptive_wavelet_model as p5
importlib.reload(p5)

print("Imported p5 from:", RUNNER_PATH)
print("DEVICE:", p5.DEVICE)
print("Available datasets:", p5.ALL_DATASETS)

ROOT: /data/Sajjan_Singh/spml/wavelet_seq_project
RUNNER_PATH: /data/Sajjan_Singh/spml/wavelet_seq_project/scripts/phase5_adaptive_wavelet_model.py
Imported p5 from: /data/Sajjan_Singh/spml/wavelet_seq_project/scripts/phase5_adaptive_wavelet_model.py
DEVICE: cuda
Available datasets: ['etth1', 'etth2', 'ettm1', 'ettm2', 'weather', 'electricity', 'traffic', 'exchange', 'ili']


In [3]:
# ============================================
# STEP 1.1: Save Phase 5 checkpoints + raw test predictions
# ============================================

from pathlib import Path
import sys
import importlib
import gc
import numpy as np
import pandas as pd
import torch

ROOT = Path("/data/Sajjan_Singh/spml/wavelet_seq_project")
SCRIPTS_DIR = ROOT / "scripts"
RUNNER_PATH = SCRIPTS_DIR / "phase5_adaptive_wavelet_model.py"

if not RUNNER_PATH.exists():
    raise FileNotFoundError(f"Missing runner file: {RUNNER_PATH}")

if str(SCRIPTS_DIR) not in sys.path:
    sys.path.append(str(SCRIPTS_DIR))

import phase5_adaptive_wavelet_model as p5
importlib.reload(p5)

PHASE5_DIR = ROOT / "results" / "tables" / "phase5"
PHASE5_CSV = PHASE5_DIR / "phase5_adaptive_wavelet_mixer_metrics.csv"

CKPT_DIR = ROOT / "results" / "checkpoints" / "phase5"
PRED_DIR = ROOT / "results" / "predictions" / "phase5"

CKPT_DIR.mkdir(parents=True, exist_ok=True)
PRED_DIR.mkdir(parents=True, exist_ok=True)

phase5_df = pd.read_csv(PHASE5_CSV)
ok = phase5_df[phase5_df["status"].astype(str) == "ok"].copy()

def row_to_cfg(row):
    return {
        "dataset": row["dataset"],
        "input_mode": row.get("input_mode", "multivariate"),
        "horizon_mode": row.get("horizon_mode", "long"),
        "lookback_mode": row.get("lookback_mode", "searched"),
        "wavelet_family": row.get("wavelet_family", "Db4"),
        "levels": int(row.get("levels", 3)),
        "d_model": int(row.get("d_model", 64)),
        "num_backbone_blocks": int(row.get("num_backbone_blocks", 3)),
        "dropout": float(row.get("dropout", 0.1)),
        "filter_reg_lambda": float(row.get("filter_reg_lambda", 1e-4)),
        "gate_entropy_lambda": float(row.get("gate_entropy_lambda", 1e-4)),
        "batch_size": int(row.get("batch_size", p5.DEFAULT_BATCH_SIZE[row["dataset"]])),
        "epochs": int(row.get("epochs", p5.DEFAULT_EPOCHS[row["dataset"]])),
    }

@torch.no_grad()
def save_phase5_preds(model, loader, target_mean, target_std, pred_path):
    model.eval()
    preds, trues = [], []
    for batch in loader:
        x = batch["x_scaled"].to(p5.DEVICE, non_blocking=True)
        y_raw = batch["y_raw"].cpu().numpy()
        with torch.cuda.amp.autocast(enabled=p5.TRAIN_CFG["use_amp"]):
            pred_scaled, aux = model(x)
        pred_raw = pred_scaled.detach().cpu().numpy() * target_std + target_mean
        preds.append(pred_raw)
        trues.append(y_raw)
    preds = np.concatenate(preds, axis=0)
    trues = np.concatenate(trues, axis=0)
    np.savez_compressed(pred_path, preds=preds, trues=trues)

registry_rows = []

for _, row in ok.iterrows():
    cfg = row_to_cfg(row)
    ds = cfg["dataset"]

    print("\n" + "=" * 100)
    print("SAVING PHASE5 MODEL:", ds)
    print("=" * 100)

    bundle = p5.load_bundle(ds, input_mode=cfg["input_mode"])
    pred_len = p5.resolve_pred_len(ds, cfg["horizon_mode"])

    if "chosen_seq_len" in row and pd.notna(row["chosen_seq_len"]):
        seq_len = int(row["chosen_seq_len"])
    else:
        seq_len = p5.resolve_seq_candidates(ds, cfg["lookback_mode"])[1]

    train_loader, val_loader, test_loader = p5.make_loaders(bundle, seq_len, pred_len, cfg["batch_size"])
    target_idx = bundle["target_idx"]
    target_mean = float(bundle["scaler_mean"][target_idx])
    target_std = float(bundle["scaler_std"][target_idx])

    model = p5.AdaptiveWaveletMixer(
        input_dim=int(bundle["values_scaled"].shape[1]),
        pred_len=pred_len,
        d_model=int(cfg["d_model"]),
        levels=int(cfg["levels"]),
        wavelet_family=cfg["wavelet_family"],
        num_backbone_blocks=int(cfg["num_backbone_blocks"]),
        dropout=float(cfg["dropout"]),
        filter_reg_lambda=float(cfg["filter_reg_lambda"]),
        gate_entropy_lambda=float(cfg["gate_entropy_lambda"]),
    ).to(p5.DEVICE)

    model, best_val = p5.train_model(
        model=model,
        train_loader=train_loader,
        val_loader=val_loader,
        target_mean=target_mean,
        target_std=target_std,
        max_epochs=int(cfg["epochs"]),
        patience=p5.TRAIN_CFG["full_patience"],
    )

    ckpt_path = CKPT_DIR / f"{ds}_adaptive_wavelet_mixer.pt"
    pred_path = PRED_DIR / f"{ds}_adaptive_wavelet_mixer_test_predictions.npz"

    torch.save({
        "dataset": ds,
        "cfg": cfg,
        "seq_len": seq_len,
        "pred_len": pred_len,
        "state_dict": model.state_dict(),
    }, ckpt_path)

    save_phase5_preds(model, test_loader, target_mean, target_std, pred_path)

    arr = np.load(pred_path)
    preds = arr["preds"]
    trues = arr["trues"]

    registry_rows.append({
        "dataset": ds,
        "model": "AdaptiveWaveletMixer",
        "family": "wavelet_adaptive",
        "seq_len": seq_len,
        "pred_len": pred_len,
        "checkpoint": str(ckpt_path.relative_to(ROOT)),
        "prediction_file": str(pred_path.relative_to(ROOT)),
        "test_mse": float(np.mean((preds - trues) ** 2)),
        "test_mae": float(np.mean(np.abs(preds - trues))),
        "test_rmse": float(np.sqrt(np.mean((preds - trues) ** 2))),
    })

    del model
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

registry_df = pd.DataFrame(registry_rows)
registry_path = PHASE5_DIR / "phase5_prediction_registry.csv"
registry_df.to_csv(registry_path, index=False)

print("\nSaved:", registry_path)
display(registry_df)


SAVING PHASE5 MODEL: etth1


/data/Sajjan_Singh/spml/wavelet_seq_project/scripts/phase5_adaptive_wavelet_model.py:567: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=TRAIN_CFG["use_amp"])
/data/Sajjan_Singh/spml/wavelet_seq_project/scripts/phase5_adaptive_wavelet_model.py:582: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=TRAIN_CFG["use_amp"]):
/data/Sajjan_Singh/spml/wavelet_seq_project/scripts/phase5_adaptive_wavelet_model.py:547: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=TRAIN_CFG["use_amp"]):
/tmp/ipykernel_1126042/224565621.py:62: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  w


SAVING PHASE5 MODEL: etth2

SAVING PHASE5 MODEL: ettm1

SAVING PHASE5 MODEL: ettm2

SAVING PHASE5 MODEL: weather

SAVING PHASE5 MODEL: electricity

SAVING PHASE5 MODEL: traffic

SAVING PHASE5 MODEL: exchange

SAVING PHASE5 MODEL: ili

Saved: /data/Sajjan_Singh/spml/wavelet_seq_project/results/tables/phase5/phase5_prediction_registry.csv


,dataset,model,family,seq_len,pred_len,checkpoint,prediction_file,test_mse,test_mae,test_rmse
0,etth1,AdaptiveWaveletMixer,wavelet_adaptive,168,96,results/checkpoints/phase5/etth1_adaptive_wave...,results/predictions/phase5/etth1_adaptive_wave...,2.620580e+01,4.640618,5.119160
1,etth2,AdaptiveWaveletMixer,wavelet_adaptive,96,96,results/checkpoints/phase5/etth2_adaptive_wave...,results/predictions/phase5/etth2_adaptive_wave...,3.253576e+02,17.058489,18.037672
2,ettm1,AdaptiveWaveletMixer,wavelet_adaptive,336,96,results/checkpoints/phase5/ettm1_adaptive_wave...,results/predictions/phase5/ettm1_adaptive_wave...,1.422699e+01,3.184678,3.771868
3,ettm2,AdaptiveWaveletMixer,wavelet_adaptive,96,96,results/checkpoints/phase5/ettm2_adaptive_wave...,results/predictions/phase5/ettm2_adaptive_wave...,2.864247e+01,4.252888,5.351866
4,weather,AdaptiveWaveletMixer,wavelet_adaptive,96,96,results/checkpoints/phase5/weather_adaptive_wa...,results/predictions/phase5/weather_adaptive_wa...,9.693031e+02,26.349178,31.133633
5,electricity,AdaptiveWaveletMixer,wavelet_adaptive,96,96,results/checkpoints/phase5/electricity_adaptiv...,results/predictions/phase5/electricity_adaptiv...,1.270459e+05,265.737671,356.435028
6,traffic,AdaptiveWaveletMixer,wavelet_adaptive,336,96,results/checkpoints/phase5/traffic_adaptive_wa...,results/predictions/phase5/traffic_adaptive_wa...,1.239006e-04,0.007849,0.011131
7,exchange,AdaptiveWaveletMixer,wavelet_adaptive,48,96,results/checkpoints/phase5/exchange_adaptive_w...,results/predictions/phase5/exchange_adaptive_w...,9.422276e-03,0.082751,0.097068
8,ili,AdaptiveWaveletMixer,wavelet_adaptive,26,24,results/checkpoints/phase5/ili_adaptive_wavele...,results/predictions/phase5/ili_adaptive_wavele...,2.145392e+11,396601.781250,463183.750000


In [4]:
# ============================================
# STEP 1.2: Build Phase 5 architecture ablation dataframe
# ============================================

import pandas as pd

REP_DATASETS = ["etth1", "weather", "traffic"]

rows = []

# A) levels
for ds in REP_DATASETS:
    for levels in [2, 3, 4]:
        rows.append({
            "ablation_name": "levels",
            "dataset": ds,
            "input_mode": "multivariate",
            "horizon_mode": "long",
            "lookback_mode": "searched",
            "wavelet_family": "Db4",
            "levels": levels,
            "d_model": 64,
            "num_backbone_blocks": 3,
            "dropout": 0.1,
            "filter_reg_lambda": 1e-4,
            "gate_entropy_lambda": 1e-4,
            "batch_size": p5.DEFAULT_BATCH_SIZE[ds],
            "epochs": p5.DEFAULT_EPOCHS[ds],
        })

# B) wavelet family
for ds in REP_DATASETS:
    for fam in ["Db2", "Db4", "Symlet"]:
        rows.append({
            "ablation_name": "family",
            "dataset": ds,
            "input_mode": "multivariate",
            "horizon_mode": "long",
            "lookback_mode": "searched",
            "wavelet_family": fam,
            "levels": 3,
            "d_model": 64,
            "num_backbone_blocks": 3,
            "dropout": 0.1,
            "filter_reg_lambda": 1e-4,
            "gate_entropy_lambda": 1e-4,
            "batch_size": p5.DEFAULT_BATCH_SIZE[ds],
            "epochs": p5.DEFAULT_EPOCHS[ds],
        })

# C) model width
for ds in REP_DATASETS:
    for d_model in [64, 128]:
        rows.append({
            "ablation_name": "width",
            "dataset": ds,
            "input_mode": "multivariate",
            "horizon_mode": "long",
            "lookback_mode": "searched",
            "wavelet_family": "Db4",
            "levels": 3,
            "d_model": d_model,
            "num_backbone_blocks": 3,
            "dropout": 0.1,
            "filter_reg_lambda": 1e-4,
            "gate_entropy_lambda": 1e-4,
            "batch_size": p5.DEFAULT_BATCH_SIZE[ds],
            "epochs": p5.DEFAULT_EPOCHS[ds],
        })

# D) backbone depth
for ds in REP_DATASETS:
    for blocks in [2, 3, 4]:
        rows.append({
            "ablation_name": "depth",
            "dataset": ds,
            "input_mode": "multivariate",
            "horizon_mode": "long",
            "lookback_mode": "searched",
            "wavelet_family": "Db4",
            "levels": 3,
            "d_model": 64,
            "num_backbone_blocks": blocks,
            "dropout": 0.1,
            "filter_reg_lambda": 1e-4,
            "gate_entropy_lambda": 1e-4,
            "batch_size": p5.DEFAULT_BATCH_SIZE[ds],
            "epochs": p5.DEFAULT_EPOCHS[ds],
        })

phase5_arch_df = pd.DataFrame(rows)
display(phase5_arch_df.head(20))
print("Total architecture-ablation experiments:", len(phase5_arch_df))

,ablation_name,dataset,input_mode,horizon_mode,lookback_mode,wavelet_family,levels,d_model,num_backbone_blocks,dropout,filter_reg_lambda,gate_entropy_lambda,batch_size,epochs
0,levels,etth1,multivariate,long,searched,Db4,2,64,3,0.1,0.0001,0.0001,256,30
1,levels,etth1,multivariate,long,searched,Db4,3,64,3,0.1,0.0001,0.0001,256,30
2,levels,etth1,multivariate,long,searched,Db4,4,64,3,0.1,0.0001,0.0001,256,30
3,levels,weather,multivariate,long,searched,Db4,2,64,3,0.1,0.0001,0.0001,192,30
4,levels,weather,multivariate,long,searched,Db4,3,64,3,0.1,0.0001,0.0001,192,30
5,levels,weather,multivariate,long,searched,Db4,4,64,3,0.1,0.0001,0.0001,192,30
6,levels,traffic,multivariate,long,searched,Db4,2,64,3,0.1,0.0001,0.0001,96,30
7,levels,traffic,multivariate,long,searched,Db4,3,64,3,0.1,0.0001,0.0001,96,30
8,levels,traffic,multivariate,long,searched,Db4,4,64,3,0.1,0.0001,0.0001,96,30
9,family,etth1,multivariate,long,searched,Db2,3,64,3,0.1,0.0001,0.0001,256,30


Total architecture-ablation experiments: 33


In [5]:
# ============================================
# STEP 1.3: Run Phase 5 architecture ablation
# ============================================

phase5_arch_results = p5.run_experiment_grid(
    exp_df=phase5_arch_df,
    out_csv_name="phase5_architecture_ablation_metrics.csv",
    overwrite=False,
)

print("Rows:", len(phase5_arch_results))
display(phase5_arch_results.tail(10))

Rows: 33


,ablation_name,dataset,input_mode,horizon_mode,lookback_mode,wavelet_family,levels,d_model,num_backbone_blocks,dropout,...,input_dim,search_best_val_rmse,val_mse,val_mae,val_rmse,test_mse,test_mae,test_rmse,train_seconds,status
23,width,traffic,multivariate,long,searched,Db4,3,128,3,0.1,...,862,0.006284,0.000031,0.003874,0.005533,0.000077,0.005601,0.008787,152.158874,ok
24,depth,etth1,multivariate,long,searched,Db4,3,64,2,0.1,...,7,3.424782,11.278936,2.711972,3.358413,27.373446,4.726367,5.231964,8.253814,ok
25,depth,etth1,multivariate,long,searched,Db4,3,64,3,0.1,...,7,3.379965,13.994154,3.057286,3.740876,29.582564,4.912198,5.438986,8.826449,ok
26,depth,etth1,multivariate,long,searched,Db4,3,64,4,0.1,...,7,3.644376,12.473704,2.921515,3.531813,26.849357,4.629353,5.181637,9.541155,ok
27,depth,weather,multivariate,long,searched,Db4,3,64,2,0.1,...,21,19.923722,363.011856,14.007330,19.052870,398.328605,14.449262,19.958171,170.406307,ok
28,depth,weather,multivariate,long,searched,Db4,3,64,3,0.1,...,21,20.383592,625.988358,16.706664,25.019759,619.521818,19.383595,24.890195,48.991362,ok
29,depth,weather,multivariate,long,searched,Db4,3,64,4,0.1,...,21,21.807149,10565.305813,25.278230,102.787673,666.742349,20.402070,25.821355,50.120693,ok
30,depth,traffic,multivariate,long,searched,Db4,3,64,2,0.1,...,862,0.006574,0.000035,0.004343,0.005954,0.000088,0.006285,0.009403,102.639376,ok
31,depth,traffic,multivariate,long,searched,Db4,3,64,3,0.1,...,862,0.006781,0.000043,0.004758,0.006525,0.000099,0.006835,0.009966,199.482447,ok
32,depth,traffic,multivariate,long,searched,Db4,3,64,4,0.1,...,862,0.006775,0.000034,0.004194,0.005835,0.000076,0.005787,0.008742,150.053496,ok


In [6]:
# ============================================
# STEP 1.4: Summarize Phase 5 architecture ablation
# ============================================

from pathlib import Path
import pandas as pd

PHASE5_ARCH_PATH = ROOT / "results" / "tables" / "phase5" / "phase5_architecture_ablation_metrics.csv"
arch_df = pd.read_csv(PHASE5_ARCH_PATH)
arch_ok = arch_df[arch_df["status"].astype(str) == "ok"].copy()

print("Usable architecture-ablation rows:", len(arch_ok))

for ablation_name in sorted(arch_ok["ablation_name"].unique()):
    sub = arch_ok[arch_ok["ablation_name"] == ablation_name].copy()

    group_cols = {
        "levels": ["levels"],
        "family": ["wavelet_family"],
        "width": ["d_model"],
        "depth": ["num_backbone_blocks"],
    }[ablation_name]

    avg = (
        sub.groupby(group_cols, as_index=False)[["test_mse", "test_mae", "test_rmse"]]
           .mean()
           .sort_values("test_rmse")
           .reset_index(drop=True)
    )

    out_path = ROOT / "results" / "tables" / "phase5" / f"summary_phase5_{ablation_name}.csv"
    avg.to_csv(out_path, index=False)

    print("\nAblation:", ablation_name)
    print("Saved:", out_path)
    display(avg)

Usable architecture-ablation rows: 33

Ablation: depth
Saved: /data/Sajjan_Singh/spml/wavelet_seq_project/results/tables/phase5/summary_phase5_depth.csv


,num_backbone_blocks,test_mse,test_mae,test_rmse
0,2,141.900713,6.393971,8.399846
1,3,216.368160,8.100876,10.113049
2,4,231.197261,8.345737,10.337244



Ablation: family
Saved: /data/Sajjan_Singh/spml/wavelet_seq_project/results/tables/phase5/summary_phase5_family.csv


,wavelet_family,test_mse,test_mae,test_rmse
0,Db4,216.368160,8.100876,10.113049
1,Db2,278.490758,9.013794,11.279488
2,Symlet,2128.576215,7.851454,28.374833



Ablation: levels
Saved: /data/Sajjan_Singh/spml/wavelet_seq_project/results/tables/phase5/summary_phase5_levels.csv


,levels,test_mse,test_mae,test_rmse
0,2,110.834669,6.300967,7.792918
1,4,166.720285,7.245109,9.285625
2,3,216.368160,8.100876,10.113049



Ablation: width
Saved: /data/Sajjan_Singh/spml/wavelet_seq_project/results/tables/phase5/summary_phase5_width.csv


,d_model,test_mse,test_mae,test_rmse
0,128,121.197029,5.823263,7.699728
1,64,216.368160,8.100876,10.113049


In [1]:
# ============================================
# STEP 1A: Setup for best-config Phase 5 rerun
# ============================================

from pathlib import Path
import sys
import importlib
import pandas as pd

ROOT = Path("/data/Sajjan_Singh/spml/wavelet_seq_project")
SCRIPTS_DIR = ROOT / "scripts"
RUNNER_PATH = SCRIPTS_DIR / "phase5_adaptive_wavelet_model.py"

if not RUNNER_PATH.exists():
    raise FileNotFoundError(f"Missing runner file: {RUNNER_PATH}")

if str(SCRIPTS_DIR) not in sys.path:
    sys.path.append(str(SCRIPTS_DIR))

import phase5_adaptive_wavelet_model as p5
importlib.reload(p5)

print("Imported p5 from:", RUNNER_PATH)
print("DEVICE:", p5.DEVICE)
print("DATASETS:", p5.ALL_DATASETS)

Imported p5 from: /data/Sajjan_Singh/spml/wavelet_seq_project/scripts/phase5_adaptive_wavelet_model.py
DEVICE: cuda
DATASETS: ['etth1', 'etth2', 'ettm1', 'ettm2', 'weather', 'electricity', 'traffic', 'exchange', 'ili']


In [2]:
# ============================================
# STEP 1B: Build best-config Phase 5 grid
# ============================================

BESTCFG_ROWS = []

for ds in p5.ALL_DATASETS:
    BESTCFG_ROWS.append({
        "dataset": ds,
        "input_mode": "multivariate",
        "horizon_mode": "long",
        "lookback_mode": "searched",
        "wavelet_family": "Db4",
        "levels": 2,
        "d_model": 128,
        "num_backbone_blocks": 2,
        "dropout": 0.1,
        "filter_reg_lambda": 1e-4,
        "gate_entropy_lambda": 1e-4,
        "batch_size": p5.DEFAULT_BATCH_SIZE[ds],
        "epochs": p5.DEFAULT_EPOCHS[ds],
    })

phase5_bestcfg_df = pd.DataFrame(BESTCFG_ROWS)
display(phase5_bestcfg_df)
print("Total experiments:", len(phase5_bestcfg_df))

,dataset,input_mode,horizon_mode,lookback_mode,wavelet_family,levels,d_model,num_backbone_blocks,dropout,filter_reg_lambda,gate_entropy_lambda,batch_size,epochs
0,etth1,multivariate,long,searched,Db4,2,128,2,0.1,0.0001,0.0001,256,30
1,etth2,multivariate,long,searched,Db4,2,128,2,0.1,0.0001,0.0001,256,30
2,ettm1,multivariate,long,searched,Db4,2,128,2,0.1,0.0001,0.0001,256,30
3,ettm2,multivariate,long,searched,Db4,2,128,2,0.1,0.0001,0.0001,256,30
4,weather,multivariate,long,searched,Db4,2,128,2,0.1,0.0001,0.0001,192,30
5,electricity,multivariate,long,searched,Db4,2,128,2,0.1,0.0001,0.0001,128,30
6,traffic,multivariate,long,searched,Db4,2,128,2,0.1,0.0001,0.0001,96,30
7,exchange,multivariate,long,searched,Db4,2,128,2,0.1,0.0001,0.0001,256,30
8,ili,multivariate,long,searched,Db4,2,128,2,0.1,0.0001,0.0001,64,35


Total experiments: 9


In [3]:
# ============================================
# STEP 1C: Run best-config Phase 5 on all 9 datasets
# ============================================

phase5_bestcfg_results = p5.run_experiment_grid(
    exp_df=phase5_bestcfg_df,
    out_csv_name="phase5_bestcfg_metrics.csv",
    overwrite=False,
)

print("Rows:", len(phase5_bestcfg_results))
display(phase5_bestcfg_results.tail(10))

Running 1/9 | etth1__multivariate__long__searched__Db4__L2__dm128__blk2


/data/Sajjan_Singh/spml/wavelet_seq_project/scripts/phase5_adaptive_wavelet_model.py:567: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=TRAIN_CFG["use_amp"])
/data/Sajjan_Singh/spml/wavelet_seq_project/scripts/phase5_adaptive_wavelet_model.py:582: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=TRAIN_CFG["use_amp"]):
/data/Sajjan_Singh/spml/wavelet_seq_project/scripts/phase5_adaptive_wavelet_model.py:547: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=TRAIN_CFG["use_amp"]):


Running 2/9 | etth2__multivariate__long__searched__Db4__L2__dm128__blk2
Running 3/9 | ettm1__multivariate__long__searched__Db4__L2__dm128__blk2
Running 4/9 | ettm2__multivariate__long__searched__Db4__L2__dm128__blk2
Running 5/9 | weather__multivariate__long__searched__Db4__L2__dm128__blk2
Running 6/9 | electricity__multivariate__long__searched__Db4__L2__dm128__blk2
Running 7/9 | traffic__multivariate__long__searched__Db4__L2__dm128__blk2
Running 8/9 | exchange__multivariate__long__searched__Db4__L2__dm128__blk2
Running 9/9 | ili__multivariate__long__searched__Db4__L2__dm128__blk2
Rows: 9


,dataset,input_mode,horizon_mode,lookback_mode,wavelet_family,levels,d_model,num_backbone_blocks,dropout,filter_reg_lambda,...,input_dim,search_best_val_rmse,val_mse,val_mae,val_rmse,test_mse,test_mae,test_rmse,train_seconds,status
0,etth1,multivariate,long,searched,Db4,2,128,2,0.1,0.0001,...,7,3.805625,1.312640e+01,2.905869,3.623038,5.523632e+01,6.703727,7.432114,9.706602,ok
1,etth2,multivariate,long,searched,Db4,2,128,2,0.1,0.0001,...,7,6.849211,4.664551e+01,5.590334,6.829752,2.638520e+02,15.162946,16.243521,5.831062,ok
2,ettm1,multivariate,long,searched,Db4,2,128,2,0.1,0.0001,...,7,2.762749,6.727039e+00,2.028607,2.593654,9.906407e+00,2.595118,3.147445,19.808688,ok
3,ettm2,multivariate,long,searched,Db4,2,128,2,0.1,0.0001,...,7,5.197466,2.828835e+01,4.084210,5.318680,7.727222e+01,7.584376,8.790462,34.600202,ok
4,weather,multivariate,long,searched,Db4,2,128,2,0.1,0.0001,...,21,21.614904,4.179320e+02,15.012554,20.443385,3.744995e+02,13.774545,19.351990,85.668875,ok
5,electricity,multivariate,long,searched,Db4,2,128,2,0.1,0.0001,...,321,280.064889,7.896415e+04,200.574456,281.005601,1.136201e+05,248.957629,337.075795,26.924991,ok
6,traffic,multivariate,long,searched,Db4,2,128,2,0.1,0.0001,...,862,0.006291,3.125175e-05,0.003996,0.005590,8.153991e-05,0.005957,0.009030,68.386852,ok
7,exchange,multivariate,long,searched,Db4,2,128,2,0.1,0.0001,...,8,0.065152,6.266023e-03,0.069736,0.079158,7.217607e-03,0.068918,0.084957,10.149824,ok
8,ili,multivariate,long,searched,Db4,2,128,2,0.1,0.0001,...,7,85936.257349,3.636243e+09,46177.300887,60301.268135,1.845133e+11,369157.179519,429550.163698,10.089167,ok


In [4]:
# ============================================
# STEP 1D: Save checkpoints + raw predictions for best-config Phase 5
# ============================================

from pathlib import Path
import gc
import numpy as np
import pandas as pd
import torch

PHASE5_DIR = ROOT / "results" / "tables" / "phase5"
PHASE5_CSV = PHASE5_DIR / "phase5_bestcfg_metrics.csv"

CKPT_DIR = ROOT / "results" / "checkpoints" / "phase5_bestcfg"
PRED_DIR = ROOT / "results" / "predictions" / "phase5_bestcfg"
CKPT_DIR.mkdir(parents=True, exist_ok=True)
PRED_DIR.mkdir(parents=True, exist_ok=True)

phase5_df = pd.read_csv(PHASE5_CSV)
ok = phase5_df[phase5_df["status"].astype(str) == "ok"].copy()

def row_to_cfg(row):
    return {
        "dataset": row["dataset"],
        "input_mode": row.get("input_mode", "multivariate"),
        "horizon_mode": row.get("horizon_mode", "long"),
        "lookback_mode": row.get("lookback_mode", "searched"),
        "wavelet_family": row.get("wavelet_family", "Db4"),
        "levels": int(row.get("levels", 2)),
        "d_model": int(row.get("d_model", 128)),
        "num_backbone_blocks": int(row.get("num_backbone_blocks", 2)),
        "dropout": float(row.get("dropout", 0.1)),
        "filter_reg_lambda": float(row.get("filter_reg_lambda", 1e-4)),
        "gate_entropy_lambda": float(row.get("gate_entropy_lambda", 1e-4)),
        "batch_size": int(row.get("batch_size", p5.DEFAULT_BATCH_SIZE[row["dataset"]])),
        "epochs": int(row.get("epochs", p5.DEFAULT_EPOCHS[row["dataset"]])),
    }

@torch.no_grad()
def save_phase5_preds(model, loader, target_mean, target_std, pred_path):
    model.eval()
    preds, trues = [], []
    for batch in loader:
        x = batch["x_scaled"].to(p5.DEVICE, non_blocking=True)
        y_raw = batch["y_raw"].cpu().numpy()
        with torch.cuda.amp.autocast(enabled=p5.TRAIN_CFG["use_amp"]):
            pred_scaled, aux = model(x)
        pred_raw = pred_scaled.detach().cpu().numpy() * target_std + target_mean
        preds.append(pred_raw)
        trues.append(y_raw)
    preds = np.concatenate(preds, axis=0)
    trues = np.concatenate(trues, axis=0)
    np.savez_compressed(pred_path, preds=preds, trues=trues)

registry_rows = []

for _, row in ok.iterrows():
    cfg = row_to_cfg(row)
    ds = cfg["dataset"]

    print("\n" + "=" * 100)
    print("SAVING BESTCFG PHASE5 MODEL:", ds)
    print("=" * 100)

    bundle = p5.load_bundle(ds, input_mode=cfg["input_mode"])
    pred_len = p5.resolve_pred_len(ds, cfg["horizon_mode"])

    if "chosen_seq_len" in row and pd.notna(row["chosen_seq_len"]):
        seq_len = int(row["chosen_seq_len"])
    else:
        seq_len = p5.resolve_seq_candidates(ds, cfg["lookback_mode"])[1]

    train_loader, val_loader, test_loader = p5.make_loaders(bundle, seq_len, pred_len, cfg["batch_size"])
    target_idx = bundle["target_idx"]
    target_mean = float(bundle["scaler_mean"][target_idx])
    target_std = float(bundle["scaler_std"][target_idx])

    model = p5.AdaptiveWaveletMixer(
        input_dim=int(bundle["values_scaled"].shape[1]),
        pred_len=pred_len,
        d_model=int(cfg["d_model"]),
        levels=int(cfg["levels"]),
        wavelet_family=cfg["wavelet_family"],
        num_backbone_blocks=int(cfg["num_backbone_blocks"]),
        dropout=float(cfg["dropout"]),
        filter_reg_lambda=float(cfg["filter_reg_lambda"]),
        gate_entropy_lambda=float(cfg["gate_entropy_lambda"]),
    ).to(p5.DEVICE)

    model, best_val = p5.train_model(
        model=model,
        train_loader=train_loader,
        val_loader=val_loader,
        target_mean=target_mean,
        target_std=target_std,
        max_epochs=int(cfg["epochs"]),
        patience=p5.TRAIN_CFG["full_patience"],
    )

    ckpt_path = CKPT_DIR / f"{ds}_adaptive_wavelet_mixer_bestcfg.pt"
    pred_path = PRED_DIR / f"{ds}_adaptive_wavelet_mixer_bestcfg_test_predictions.npz"

    torch.save({
        "dataset": ds,
        "cfg": cfg,
        "seq_len": seq_len,
        "pred_len": pred_len,
        "state_dict": model.state_dict(),
    }, ckpt_path)

    save_phase5_preds(model, test_loader, target_mean, target_std, pred_path)

    arr = np.load(pred_path)
    preds = arr["preds"]
    trues = arr["trues"]

    registry_rows.append({
        "dataset": ds,
        "model": "AdaptiveWaveletMixer",
        "family": "wavelet_adaptive",
        "seq_len": seq_len,
        "pred_len": pred_len,
        "checkpoint": str(ckpt_path.relative_to(ROOT)),
        "prediction_file": str(pred_path.relative_to(ROOT)),
        "test_mse": float(np.mean((preds - trues) ** 2)),
        "test_mae": float(np.mean(np.abs(preds - trues))),
        "test_rmse": float(np.sqrt(np.mean((preds - trues) ** 2))),
    })

    del model
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

registry_df = pd.DataFrame(registry_rows)
registry_path = PHASE5_DIR / "phase5_bestcfg_prediction_registry.csv"
registry_df.to_csv(registry_path, index=False)

print("\nSaved:", registry_path)
display(registry_df)


SAVING BESTCFG PHASE5 MODEL: etth1


/data/Sajjan_Singh/spml/wavelet_seq_project/scripts/phase5_adaptive_wavelet_model.py:567: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=TRAIN_CFG["use_amp"])
/data/Sajjan_Singh/spml/wavelet_seq_project/scripts/phase5_adaptive_wavelet_model.py:582: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=TRAIN_CFG["use_amp"]):
/data/Sajjan_Singh/spml/wavelet_seq_project/scripts/phase5_adaptive_wavelet_model.py:547: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=TRAIN_CFG["use_amp"]):
/tmp/ipykernel_2523386/2004483839.py:46: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  


SAVING BESTCFG PHASE5 MODEL: etth2

SAVING BESTCFG PHASE5 MODEL: ettm1

SAVING BESTCFG PHASE5 MODEL: ettm2

SAVING BESTCFG PHASE5 MODEL: weather

SAVING BESTCFG PHASE5 MODEL: electricity

SAVING BESTCFG PHASE5 MODEL: traffic

SAVING BESTCFG PHASE5 MODEL: exchange

SAVING BESTCFG PHASE5 MODEL: ili

Saved: /data/Sajjan_Singh/spml/wavelet_seq_project/results/tables/phase5/phase5_bestcfg_prediction_registry.csv


,dataset,model,family,seq_len,pred_len,checkpoint,prediction_file,test_mse,test_mae,test_rmse
0,etth1,AdaptiveWaveletMixer,wavelet_adaptive,168,96,results/checkpoints/phase5_bestcfg/etth1_adapt...,results/predictions/phase5_bestcfg/etth1_adapt...,4.471373e+01,6.204969,6.686832
1,etth2,AdaptiveWaveletMixer,wavelet_adaptive,48,96,results/checkpoints/phase5_bestcfg/etth2_adapt...,results/predictions/phase5_bestcfg/etth2_adapt...,2.213284e+02,13.749595,14.877110
2,ettm1,AdaptiveWaveletMixer,wavelet_adaptive,96,96,results/checkpoints/phase5_bestcfg/ettm1_adapt...,results/predictions/phase5_bestcfg/ettm1_adapt...,1.933618e+01,3.962334,4.397292
3,ettm2,AdaptiveWaveletMixer,wavelet_adaptive,96,96,results/checkpoints/phase5_bestcfg/ettm2_adapt...,results/predictions/phase5_bestcfg/ettm2_adapt...,6.362106e+01,6.937624,7.976281
4,weather,AdaptiveWaveletMixer,wavelet_adaptive,288,96,results/checkpoints/phase5_bestcfg/weather_ada...,results/predictions/phase5_bestcfg/weather_ada...,3.754623e+02,14.075797,19.376850
5,electricity,AdaptiveWaveletMixer,wavelet_adaptive,96,96,results/checkpoints/phase5_bestcfg/electricity...,results/predictions/phase5_bestcfg/electricity...,1.224819e+05,261.316162,349.974152
6,traffic,AdaptiveWaveletMixer,wavelet_adaptive,96,96,results/checkpoints/phase5_bestcfg/traffic_ada...,results/predictions/phase5_bestcfg/traffic_ada...,8.496914e-05,0.005922,0.009218
7,exchange,AdaptiveWaveletMixer,wavelet_adaptive,96,96,results/checkpoints/phase5_bestcfg/exchange_ad...,results/predictions/phase5_bestcfg/exchange_ad...,8.972298e-03,0.077934,0.094722
8,ili,AdaptiveWaveletMixer,wavelet_adaptive,52,24,results/checkpoints/phase5_bestcfg/ili_adaptiv...,results/predictions/phase5_bestcfg/ili_adaptiv...,2.191652e+11,405056.343750,468150.843750


In [ ]:
# ============================================
# RUN NEXT: Full rerun of PBT-discovered best config on all 9 datasets
# Paste at the end of adaptive_wavelet_model.ipynb
# ============================================

from pathlib import Path
import sys
import importlib
import gc
import json
import numpy as np
import pandas as pd
import torch

ROOT = Path("/data/Sajjan_Singh/spml/wavelet_seq_project").resolve()
SCRIPTS_DIR = ROOT / "scripts"
RUNNER_PATH = SCRIPTS_DIR / "phase5_adaptive_wavelet_model.py"

if str(SCRIPTS_DIR) not in sys.path:
    sys.path.append(str(SCRIPTS_DIR))

import phase5_adaptive_wavelet_model as p5
importlib.reload(p5)

SEARCH_DIR = ROOT / "results" / "pbt_search"
RERUN_GRID_PATH = SEARCH_DIR / "pbt_bestcfg_full_rerun_grid_fixed.csv"

if not RERUN_GRID_PATH.exists():
    raise FileNotFoundError(f"Missing rerun grid: {RERUN_GRID_PATH}")

PHASE5_DIR = ROOT / "results" / "tables" / "phase5"
PHASE5_DIR.mkdir(parents=True, exist_ok=True)

CKPT_DIR = ROOT / "results" / "checkpoints" / "phase5_pbt_bestcfg"
PRED_DIR = ROOT / "results" / "predictions" / "phase5_pbt_bestcfg"
CKPT_DIR.mkdir(parents=True, exist_ok=True)
PRED_DIR.mkdir(parents=True, exist_ok=True)

print("Using rerun grid:", RERUN_GRID_PATH)
rerun_df = pd.read_csv(RERUN_GRID_PATH)
display(rerun_df)

# ------------------------------------------------
# 1) Full clean rerun
# ------------------------------------------------
phase5_pbt_results = p5.run_experiment_grid(
    exp_df=rerun_df,
    out_csv_name="phase5_pbt_bestcfg_metrics.csv",
    overwrite=False,
)

print("\nSaved metrics CSV to:",
      ROOT / "results" / "tables" / "phase5" / "phase5_pbt_bestcfg_metrics.csv")
display(phase5_pbt_results.tail(10))

# ------------------------------------------------
# 2) Save checkpoints + raw predictions
# ------------------------------------------------
PHASE5_CSV = PHASE5_DIR / "phase5_pbt_bestcfg_metrics.csv"
phase5_df = pd.read_csv(PHASE5_CSV)
ok = phase5_df[phase5_df["status"].astype(str) == "ok"].copy()

def row_to_cfg(row):
    return {
        "dataset": row["dataset"],
        "input_mode": row.get("input_mode", "multivariate"),
        "horizon_mode": row.get("horizon_mode", "long"),
        "lookback_mode": row.get("lookback_mode", "searched"),
        "wavelet_family": row.get("wavelet_family", "Db2"),
        "levels": int(row.get("levels", 3)),
        "d_model": int(row.get("d_model", 64)),
        "num_backbone_blocks": int(row.get("num_backbone_blocks", 1)),
        "dropout": float(row.get("dropout", 0.05)),
        "filter_reg_lambda": float(row.get("filter_reg_lambda", 1e-5)),
        "gate_entropy_lambda": float(row.get("gate_entropy_lambda", 1e-5)),
        "batch_size": int(row.get("batch_size", p5.DEFAULT_BATCH_SIZE[row["dataset"]])),
        "epochs": int(row.get("epochs", p5.DEFAULT_EPOCHS[row["dataset"]])),
    }

@torch.no_grad()
def save_phase5_preds(model, loader, target_mean, target_std, pred_path):
    model.eval()
    preds, trues = [], []
    for batch in loader:
        x = batch["x_scaled"].to(p5.DEVICE, non_blocking=True)
        y_raw = batch["y_raw"].cpu().numpy()
        with torch.cuda.amp.autocast(enabled=p5.TRAIN_CFG["use_amp"]):
            pred_scaled, aux = model(x)
        pred_raw = pred_scaled.detach().cpu().numpy() * target_std + target_mean
        preds.append(pred_raw)
        trues.append(y_raw)
    preds = np.concatenate(preds, axis=0)
    trues = np.concatenate(trues, axis=0)
    np.savez_compressed(pred_path, preds=preds, trues=trues)

registry_rows = []

for _, row in ok.iterrows():
    cfg = row_to_cfg(row)
    ds = cfg["dataset"]

    print("\n" + "=" * 100)
    print("SAVING PBT-BESTCFG MODEL:", ds)
    print("=" * 100)

    bundle = p5.load_bundle(ds, input_mode=cfg["input_mode"])
    pred_len = p5.resolve_pred_len(ds, cfg["horizon_mode"])

    if "chosen_seq_len" in row and pd.notna(row["chosen_seq_len"]):
        seq_len = int(row["chosen_seq_len"])
    else:
        seq_len = p5.resolve_seq_candidates(ds, cfg["lookback_mode"])[1]

    train_loader, val_loader, test_loader = p5.make_loaders(bundle, seq_len, pred_len, cfg["batch_size"])
    target_idx = bundle["target_idx"]
    target_mean = float(bundle["scaler_mean"][target_idx])
    target_std = float(bundle["scaler_std"][target_idx])

    model = p5.AdaptiveWaveletMixer(
        input_dim=int(bundle["values_scaled"].shape[1]),
        pred_len=pred_len,
        d_model=int(cfg["d_model"]),
        levels=int(cfg["levels"]),
        wavelet_family=cfg["wavelet_family"],
        num_backbone_blocks=int(cfg["num_backbone_blocks"]),
        dropout=float(cfg["dropout"]),
        filter_reg_lambda=float(cfg["filter_reg_lambda"]),
        gate_entropy_lambda=float(cfg["gate_entropy_lambda"]),
    ).to(p5.DEVICE)

    model, best_val = p5.train_model(
        model=model,
        train_loader=train_loader,
        val_loader=val_loader,
        target_mean=target_mean,
        target_std=target_std,
        max_epochs=int(cfg["epochs"]),
        patience=p5.TRAIN_CFG["full_patience"],
    )

    ckpt_path = CKPT_DIR / f"{ds}_adaptive_wavelet_mixer_pbtbest.pt"
    pred_path = PRED_DIR / f"{ds}_adaptive_wavelet_mixer_pbtbest_test_predictions.npz"

    torch.save({
        "dataset": ds,
        "cfg": cfg,
        "seq_len": seq_len,
        "pred_len": pred_len,
        "state_dict": model.state_dict(),
    }, ckpt_path)

    save_phase5_preds(model, test_loader, target_mean, target_std, pred_path)

    arr = np.load(pred_path)
    preds = arr["preds"]
    trues = arr["trues"]

    registry_rows.append({
        "dataset": ds,
        "model": "AdaptiveWaveletMixer_PBT",
        "family": "wavelet_adaptive",
        "seq_len": seq_len,
        "pred_len": pred_len,
        "checkpoint": str(ckpt_path.relative_to(ROOT)),
        "prediction_file": str(pred_path.relative_to(ROOT)),
        "test_mse": float(np.mean((preds - trues) ** 2)),
        "test_mae": float(np.mean(np.abs(preds - trues))),
        "test_rmse": float(np.sqrt(np.mean((preds - trues) ** 2))),
    })

    del model
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

registry_df = pd.DataFrame(registry_rows)
registry_path = PHASE5_DIR / "phase5_pbt_bestcfg_prediction_registry.csv"
registry_df.to_csv(registry_path, index=False)

print("\nSaved prediction registry:", registry_path)
display(registry_df)